# ApexID: closed-set retrieval and representation evaluation

**Run once per model, retain the output ZIP, then use this same notebook in comparison mode.**

This notebook consumes the `embeddings/<model>/<dataset>/embeddings.npz` caches produced by
your Stage 01 extraction notebook. It reuses Stage 00's frozen `closed_role` splits.
It requires **no GPU, model inference, GitHub checkout, source images, or rejection calibrator**
for its numerical evaluation. Original images are optional for retrieval example panels.

The notebook implements:

| Evidence | Outputs |
|---|---|
| Closed-set retrieval | mAP, Rank-1/5/10/20/50, CMC, mINP; query-, identity-, and observation-balanced summaries |
| Hierarchy-related discrimination | Within-category retrieval; positive/sibling/cross-category similarities; exact hard-impostor margins |
| Identity verification | AUROC, interpolated EER, attainable TAR at FAR 1% and 0.1%, ROC curves |
| Geometry | Identity and category silhouette, within-category silhouette, Fisher trace ratios, within-ID scatter, alignment, uniformity, covariance spectrum/effective rank |
| Stress tests | 1/2/5-observation gallery; nested sibling and cross-category distractors; camera/session/laterality conditions when annotated |
| Uncertainty | Fixed-gallery query-identity-block bootstrap for retrieval and paired model differences |
| Comparison | Dataset-macro score table, per-dataset results, paired differences/CIs, identity gains, diagnostic differences, curves, and explicit training-seed summaries |

**Scope:** closed-set retrieval and representation diagnostics. There is no novelty detection,
open-set clustering, or downstream WildFusion calibration here. A hierarchy-causality claim
still requires separately trained ablation checkpoints; this notebook can evaluate those checkpoints' caches.

## Inputs and workflow

Set APEXID_PREPARED_INPUT to the prepared manifest bundle and APEXID_STAGE01_INPUT to the embedding root. Use APEXID_CLOSED_MODE=evaluate (default) to score cached embeddings. Set it to compare and list completed run folders in APEXID_CLOSED_RUNS, separated by the platform path separator, to calculate paired contrasts.

Use standard model names of the form family_objective. Families are convnext_small, miewid_v3 and megadescriptor_l. Objectives are arcface, dynamic_arcface, sphereface2 and apexid_hierarchical. Use miewid_v3_pretrained and megadescriptor_l_pretrained for the unmodified references. Published checkpoint aliases are also recognized.

## Configuration

Optionally set APEXID_CLOSED_MODELS to a comma-separated subset of descriptor names and APEXID_CLOSED_OUTPUT to the report directory. The notebook keeps all five split seeds and the four external datasets for the paper results.

In [ ]:
from pathlib import Path
import os

# MODE: evaluate one/many embedding models, compare completed runs, or both.
MODE = os.environ.get("APEXID_CLOSED_MODE", "evaluate")                       # "evaluate", "compare", "both"
STAGE01_ROOT = None
STAGE00_ROOT = None
MODELS = os.environ["APEXID_CLOSED_MODELS"].split(",") if "APEXID_CLOSED_MODELS" in os.environ else None
DATASETS = ["bristol_gorillas", "elephants", "gcn_id", "jaguar_antonio"]
SPLIT_SEEDS = None                       # None = Stage 00 protocol's frozen seeds
OUTPUT_ROOT = Path(os.environ.get("APEXID_CLOSED_OUTPUT", "outputs/closed_set"))
COMPARE_ROOTS = [Path(path) for path in os.environ.get("APEXID_CLOSED_RUNS", "").split(os.pathsep) if path]
FAMILIES = {
    "convnext_small": "ConvNeXt-Small",
    "miewid_v3": "MIEW-ID v3",
    "megadescriptor_l": "MegaDescriptor-L",
}
OBJECTIVES = {
    "arcface": "ArcFace",
    "dynamic_arcface": "Dynamic ArcFace",
    "sphereface2": "SphereFace2",
    "apexid_hierarchical": "ApexID Hierarchical",
    "pretrained": "Pretrained",
}
MODEL_METADATA = {
    f"{family_key}_{loss_key}": {"family": family, "loss": loss, "label": f"{family} / {loss}"}
    for family_key, family in FAMILIES.items()
    for loss_key, loss in OBJECTIVES.items()
    if (family_key, loss_key) != ("convnext_small", "pretrained")
}
MODEL_ALIASES = {
    "ApexID_AF": "convnext_small_arcface",
    "ApexID_DAF": "convnext_small_dynamic_arcface",
    "ApexID_SF2": "convnext_small_sphereface2",
    "Apexid_Convnext_small_bhca_scsf2_ablation_e6": "convnext_small_apexid_hierarchical",
    "ApexID_Miewid_AF": "miewid_v3_arcface",
    "ApexID_Miewid_DAF": "miewid_v3_dynamic_arcface",
    "ApexID_Miewid_SF2": "miewid_v3_sphereface2",
    "ApexID_MiewID_SCSF2_e6": "miewid_v3_apexid_hierarchical",
    "MiewID_msv3_Official": "miewid_v3_pretrained",
    "ApexID_MegaDescriptor_AF": "megadescriptor_l_arcface",
    "ApexID_MegaDescriptor_DAF": "megadescriptor_l_dynamic_arcface",
    "ApexID_MegaDescriptor_SF2": "megadescriptor_l_sphereface2",
    "ApexID_MegaDescriptor_SCSF2_e6": "megadescriptor_l_apexid_hierarchical",
    "MegaDescriptor_L_384_Official": "megadescriptor_l_pretrained",
}
MODEL_METADATA.update({alias: MODEL_METADATA[name] for alias, name in MODEL_ALIASES.items()})

# Coarse animal categories, not necessarily biological species. Used only if explicit
# category/species metadata are absent. Add mappings for additional verified datasets.
DATASET_CATEGORY_FALLBACK = {"bristol_gorillas": "gorilla", "elephants": "elephants",
                             "gcn_id": "GCN", "jaguar_antonio": "jaguar"}
CATEGORY_ALIASES = {"western lowland gorilla": "gorilla", "gorillas": "gorilla",
                    "african forest elephant": "elephants", "elephant": "elephants",
                    "great crested newt": "GCN", "gcn": "GCN", "jaguars": "jaguar"}
COLUMN_ALIASES = {
    "category": ["category", "animal_category", "species", "species_name", "taxon"],
    "observation": ["observation_group_id", "observation_group", "observation_id"],
    "camera": ["camera_id", "camera", "camera_name"],
    "session": ["session_id", "session", "capture_session"],
    "side": ["laterality", "side", "viewpoint_side"],
}
EXTRA_METADATA_CSV = None                 # Optional CSV keyed by dataset,image_id; never used to rebuild splits

RUN_PAIR_ANALYSIS = True
RUN_GEOMETRY = True
RUN_FEW_SHOT = True
RUN_DISTRACTOR_SCALING = True
RESUME_COMPLETED = True
MAKE_ZIP = True
SHOW_PLOTS = False                       # plots are always saved; True also displays them in the notebook

CONFIG = {
    "random_seed": 20261023,
    "ranks": [1, 5, 10, 20, 50],
    "cmc_max_rank": 50,
    "similarity_block": 128,            # CPU, exact ranking; never a full all-image similarity matrix
    "pair_queries_per_identity": 4,
    "pair_max_queries": 800,
    "positive_pairs_per_query": 16,
    "negative_pairs_per_query": 64,
    "far_targets": [0.01, 0.001],
    "geometry_per_identity": 8,         # at most one image per recorded observation
    "geometry_max_samples": 1500,
    "uniformity_t": 2.0,
    "few_shot_observations": [1, 2, 5],
    "distractor_counts": [0, 5, 20, 50, 100],
    "distractor_queries_per_identity": 3,
    "distractor_max_queries": 500,
    "bootstrap_replicates": 10000,
    "confidence_level": 0.95,
    "export_top_k": 10,
    "numeric_threads": min(4, os.cpu_count() or 1),
}

## 3. Metric contract

For normalized embeddings, cosine similarity is their dot product. Gallery images are sorted
by decreasing cosine; exact ties use lexicographically sorted `(dataset,image_id)` keys.

For positive gallery ranks $r_1<\cdots<r_R$:

$$\mathrm{AP}(q)=\frac1R\sum_{j=1}^{R}\frac{j}{r_j},\qquad
\mathrm{INP}(q)=\frac{R}{r_R},\qquad
\mathrm{Rank}\text{-}k(q)=\mathbb{1}[r_1\leq k].$$

Rank-$k$/CMC@$k$ inspect **gallery images**, not deduplicated identity lists. mAP and mINP
average over valid queries. Identity-balanced scores first average within identity. Observation-balanced
scores first average within the query observation group. Dataset macro first averages fixed split seeds
within each dataset and then weights the declared datasets equally; pooled analysis is excluded.

Every primary query must have a valid gallery positive. Conditional camera/session/side/category
analyses may remove all positives for a query; those queries have `valid=False`, undefined metrics,
an exclusion reason, and explicit coverage counts. They are not silently scored as failures or successes.

**Verification:** positive pairs have the same identity and different observations. Sibling negatives
have different identities within the same category. Cross-category negatives use different labeled
categories. Pairs are shared across models. Weights give equal mass to each query identity, then
each sampled query, then each sampled pair. Positive and negative classes each have total weight one.
Pairs use matched eligible anchors in each ROC comparison. EER interpolates the ROC crossing;
TAR@FAR uses an attainable threshold, without interpolation or splitting tied scores. A reported
test ROC threshold is descriptive and is not a validated deployment threshold. Small FAR estimates
can be unstable when few independent identities are available; pair counts do not measure independence.

**Hard margin:** exact mean similarity to all valid positive gallery images minus the maximum
similarity to valid impostors. Both unrestricted and sibling-only versions are saved for every query.
Identity-weighted mean, median, 10th percentile, and non-positive fraction summarize them.

**Geometry:** one image per recorded observation, with bounded identity-balanced sampling.
Silhouette uses cosine distances in the original space; singleton identities are excluded from its
final mean and coverage is reported. Effective rank is specifically
$\exp(-\sum_j p_j\log p_j)$ for normalized **centered covariance eigenvalues** $p_j$;
it is not the singular-value-entropy alternative. Fisher trace ratio is identity-balanced
between-centroid scatter divided by within-identity scatter. Uniformity is
$\log\operatorname{mean}_{i<j}\exp[-t\|z_i-z_j\|_2^2]$ with fixed $t$.
Alignment is the weighted squared distance of sampled positive pairs. Larger effective rank,
stronger category separation, or a prettier projection is not automatically better retrieval.

**Uncertainty:** percentile query-identity-block bootstrap, conditional on the evaluated galleries,
checkpoints, datasets, and fixed split seeds. One identity draw is reused across seeds; paired
comparisons reuse it for both models. Datasets are kept fixed and weighted equally. Split SD is
descriptive; it is not training-seed uncertainty. Representation diagnostics do not receive an
incorrect confidence interval that treats millions of overlapping pairs as independent.
No p-values or multiple-comparison-adjusted significance claims are generated.

### Imports and reproducibility

Dependencies: NumPy, pandas, SciPy, scikit-learn, Matplotlib, and threadpoolctl. Install them in the Jupyter environment before running. No package installation or network call is performed by this notebook.

In [ ]:
# Keep the notebook focused on the analyses reported in the paper.
RUN_POOLED = False
RUN_METADATA_SLICES = False
RUN_PCA_PLOTS = False
RUN_TSNE_PLOTS = False
EXPORT_IMAGE_PANELS = False
IMAGE_PATH_REWRITES = {}

import gc
import hashlib
import inspect
import itertools
import json
import math
import platform
import re
import shutil
import time
import warnings
from collections import Counter, defaultdict
from datetime import datetime, timezone

for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_thread_var, str(CONFIG["numeric_threads"]))
import numpy as np
import pandas as pd
import scipy
from scipy.special import logsumexp
import sklearn
from sklearn.metrics import roc_curve, roc_auc_score, silhouette_samples
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

VERSION = "apexid-embedding-evaluation-1.0.0"
IMPLEMENTATION_SHA256 = "8724219d14ae2c9eef1c63bf31460740a737a30c9c6876f177b7e17ebc9f1185"
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
METRIC_COLUMNS = ["AP", "INP", *[f"rank{k}" for k in CONFIG["ranks"]]]
MISSING_TOKENS = {"", "nan", "none", "null", "unknown", "n/a", "na", "-1", "new_whale"}

def json_clean(value):
    if isinstance(value, dict): return {str(k): json_clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_clean(v) for v in value]
    if isinstance(value, np.ndarray): return json_clean(value.tolist())
    if isinstance(value, Path): return str(value)
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.bool_,)): return bool(value)
    if isinstance(value, (float, np.floating)): return float(value) if np.isfinite(value) else None
    return value

def stable_json(value):
    return json.dumps(json_clean(value), sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def digest(value): return hashlib.sha256(stable_json(value).encode()).hexdigest()

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(2**20), b""): h.update(block)
    return h.hexdigest()

def seed_for(*parts): return int(digest(parts)[:16], 16) % (2**32)

def safe_name(text): return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("._")[:90] or "item"

def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(json_clean(obj), indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    temp.replace(path)

def save_table(path, rows, columns=None):
    frame = rows if isinstance(rows, pd.DataFrame) else pd.DataFrame(rows, columns=columns)
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if len(frame.columns): frame.to_csv(path, index=False, compression="infer")
    return frame

def read_table(path):
    try: return pd.read_csv(path, keep_default_na=False)
    except pd.errors.EmptyDataError: return pd.DataFrame()

def frame_hash(frame, columns):
    return digest(frame[columns].fillna("").astype(str).to_dict("records"))

def normalized(z):
    z = np.asarray(z, dtype=np.float32)
    if z.ndim != 2 or not len(z) or not np.isfinite(z).all():
        raise ValueError("Embeddings must be a nonempty finite N x D matrix")
    norms = np.linalg.norm(z.astype(np.float64), axis=1)
    if np.any(norms < 1e-12): raise ValueError("Zero-norm embeddings are invalid")
    if np.max(np.abs(norms - 1)) > 1e-3:
        raise ValueError("Stage 01 cache is not L2-normalized (tolerance 1e-3); check extraction")
    return (z / norms[:, None]).astype(np.float32)

def status_record(rows, dataset, seed, analysis, status, reason="", **extra):
    rows.append(dict(dataset=dataset, split_seed=seed, analysis=analysis, status=status, reason=reason, **extra))

def save_figure(fig, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(); fig.savefig(path, bbox_inches="tight")
    if SHOW_PLOTS: plt.show()
    plt.close(fig)

### Input discovery, schema checks, and observation-safe frozen roles

These helpers read only local manifests and caches. Cache rows are realigned by image identifier after checking membership and labels. Missing observation IDs fail the primary protocol. Observation identifiers are scoped by source dataset, not replaced by unique image IDs.

In [ ]:
def find_root(explicit, filename, env_name):
    supplied = explicit or os.environ.get(env_name)
    if supplied:
        p = Path(supplied).expanduser().resolve()
        if not (p / filename).is_file(): raise FileNotFoundError(f"{p} must contain {filename}")
        return p
    raise FileNotFoundError(f"Set {env_name} to the folder containing {filename}.")


def cleaned_series(series):
    result = series.fillna("").astype(str).str.strip()
    return result.mask(result.str.lower().isin(MISSING_TOKENS), "")

def canonical_category(value):
    s = str(value).strip()
    if s.lower() in MISSING_TOKENS: return ""
    return CATEGORY_ALIASES.get(s.lower(), s.lower())

def enrich_metadata(frame, extra=None):
    f = frame.copy()
    if extra is not None:
        cols = [c for c in extra if c not in {"dataset", "image_id"}]
        addon = extra.set_index(["dataset", "image_id"])
        keys = pd.MultiIndex.from_frame(f[["dataset", "image_id"]])
        aligned = addon.reindex(keys)
        for col in cols:
            values = cleaned_series(pd.Series(aligned[col].to_numpy(), index=f.index))
            if col in f:
                original = cleaned_series(f[col])
                conflict = original.ne("") & values.ne("") & original.ne(values)
                if conflict.any(): raise ValueError(f"Extra metadata conflicts with {col}")
                f[col] = original.mask(original.eq(""), values)
            else: f[col] = values
    for target, aliases in COLUMN_ALIASES.items():
        out = pd.Series("", index=f.index, dtype=object)
        # Priority is explicit: observation_group_id may differ from raw observation_id.
        for column in aliases:
            if column not in f: continue
            values = cleaned_series(f[column])
            if target == "category": values = values.map(canonical_category)
            if target == "side":
                values = values.str.lower().replace({"l": "left", "r": "right"})
                values = values.where(values.isin(["left", "right"]), "")
            out = out.mask(out.eq("") & values.ne(""), values)
        if target == "category":
            for ds, value in DATASET_CATEGORY_FALLBACK.items():
                out = out.mask(out.eq("") & f.dataset.eq(ds), canonical_category(value))
        f[target] = out
    f["key"] = [stable_json([d, i]) for d, i in zip(f.dataset, f.image_id)]
    f["identity_key"] = [stable_json([d, y]) for d, y in zip(f.dataset, f.identity)]
    f["observation_key"] = [stable_json([d, o]) if o else "" for d, o in zip(f.dataset, f.observation)]
    for identity, group in f.groupby("identity_key"):
        known = set(group.category) - {""}
        if len(known) > 1: raise ValueError(f"Conflicting category labels for {identity}: {known}")
        if known: f.loc[group.index, "category"] = next(iter(known))
    return f

def assert_frozen_split(frame):
    required = {"image_id", "identity", "dataset", "closed_role", "key", "observation_key"}
    if not required <= set(frame): raise ValueError(f"Split lacks {required - set(frame)}")
    if frame.key.duplicated().any(): raise ValueError("Duplicate image keys in split")
    if not set(frame.closed_role) <= {"gallery", "query_known", "excluded"}:
        raise ValueError("Unexpected closed_role. Use the Stage 00 closed_role column, not open_role")
    active = frame.loc[frame.closed_role.isin(["gallery", "query_known"])]
    if active.observation_key.eq("").any():
        raise ValueError("Missing observation groups in active split; never substitute per-image identifiers")
    gallery = active.loc[active.closed_role.eq("gallery")]
    query = active.loc[active.closed_role.eq("query_known")]
    if gallery.empty or query.empty: raise ValueError("Frozen closed-set gallery/query is empty")
    if set(gallery.observation_key) & set(query.observation_key):
        raise ValueError("Recorded observation groups cross gallery and query")
    if not set(query.identity_key) <= set(gallery.identity_key):
        raise ValueError("A closed-set query identity is absent from the gallery")
    if "image_sha256" in active:
        ghash = set(cleaned_series(gallery.image_sha256)) - {""}
        qhash = set(cleaned_series(query.image_sha256)) - {""}
        if ghash & qhash: raise ValueError("Exact image hashes cross gallery/query")
    return gallery.sort_values("key").reset_index(drop=True), query.sort_values("key").reset_index(drop=True)

def prepare_inputs():
    root0 = find_root(STAGE00_ROOT, "preparation_manifest.json", "APEXID_PREPARED_INPUT")
    root1 = find_root(STAGE01_ROOT, "stage01_manifest.json", "APEXID_STAGE01_INPUT")
    m0 = json.loads((root0 / "preparation_manifest.json").read_text())
    m1 = json.loads((root1 / "stage01_manifest.json").read_text())
    if m0.get("status") != "passed" or m1.get("status") != "passed":
        raise ValueError("Both Stage 00 and Stage 01 must be completed with status=passed")
    prepared_hash = m1.get("prepared_manifest_sha256")
    if prepared_hash and prepared_hash != file_sha(root0 / "preparation_manifest.json"):
        raise ValueError("Stage 01 belongs to a different prepared bundle")
    seeds = SPLIT_SEEDS if SPLIT_SEEDS is not None else m0.get("protocol", {}).get("split_seeds")
    if not seeds: raise ValueError("Set SPLIT_SEEDS explicitly; no frozen seeds found")
    seeds = sorted(set(map(int, seeds)))
    datasets = list(DATASETS)
    if not datasets or len(set(datasets)) != len(datasets): raise ValueError("DATASETS must be nonempty and unique")
    if "apexid_calibration" in datasets: raise ValueError("Training/calibration cache cannot be an external-test dataset")
    embroot = root1 / "embeddings"
    if not embroot.is_dir(): raise FileNotFoundError(f"Missing portable embeddings directory: {embroot}")
    available = sorted(p.name for p in embroot.iterdir() if p.is_dir() and any(p.glob("*/embeddings.npz")))
    models = list(MODELS) if MODELS is not None else list(m1.get("models_completed") or available)
    if not models or len(set(models)) != len(models) or set(models) - set(available):
        raise ValueError(f"MODELS must select distinct cache folders; available={available}")
    extra = None
    if EXTRA_METADATA_CSV:
        extra = pd.read_csv(EXTRA_METADATA_CSV, dtype=str).fillna("")
        if not {"dataset", "image_id"} <= set(extra) or extra.duplicated(["dataset", "image_id"]).any():
            raise ValueError("Extra metadata needs unique dataset,image_id keys")
    canon, splits, provenance, memberships = {}, {}, {}, {}
    for ds in datasets:
        path = root0 / "manifests" / f"{ds}_canonical_manifest.csv"
        f = pd.read_csv(path, dtype=str).fillna("")
        if not {"image_id", "identity", "dataset"} <= set(f) or f.image_id.duplicated().any():
            raise ValueError(f"Invalid canonical schema: {ds}")
        if set(f.dataset) != {ds} or cleaned_series(f.identity).eq("").any():
            raise ValueError(f"Invalid dataset/identity labels: {ds}")
        canon[ds] = enrich_metadata(f, extra)
        provenance[f"canonical/{ds}"] = file_sha(path)
        for seed in seeds:
            p = root0 / "splits" / ds / f"seed_{seed}" / "split_manifest.csv"
            s = pd.read_csv(p, dtype=str).fillna("")
            if s.image_id.duplicated().any() or set(s.image_id) != set(f.image_id):
                raise ValueError(f"Split and canonical memberships differ: {ds}/{seed}")
            expected = f.set_index("image_id").loc[s.image_id, "identity"].to_numpy()
            if not np.array_equal(expected, s.identity.to_numpy()): raise ValueError("Split identity labels changed")
            if "dataset" not in s: s["dataset"] = ds
            if set(s.dataset) != {ds}: raise ValueError("Wrong dataset in split")
            if "split_seed" in s and set(s.split_seed) != {str(seed)}: raise ValueError("Wrong split seed")
            # Bring missing metadata columns from canonical without overwriting frozen split fields.
            c = f.set_index("image_id").loc[s.image_id]
            for col in f.columns:
                if col not in s: s[col] = c[col].to_numpy()
            s = enrich_metadata(s, extra)
            assert_frozen_split(s)
            actual_hash = file_sha(p)
            declared = m0.get("split_manifest_hashes", {}).get(f"{ds}::seed_{seed}")
            if declared and declared != actual_hash: raise ValueError(f"Frozen split hash changed: {p}")
            splits[(ds, seed)] = s
            provenance[f"split/{ds}/{seed}"] = actual_hash
            memberships[f"{ds}/{seed}"] = frame_hash(s.sort_values("key"),
                ["key", "identity_key", "observation_key", "category", "closed_role", "camera", "session", "side"])
    protocol = {
        "version": VERSION, "implementation_sha256": IMPLEMENTATION_SHA256,
        "datasets": datasets, "seeds": seeds, "config": CONFIG,
        "pooled": RUN_POOLED, "pair": RUN_PAIR_ANALYSIS, "geometry": RUN_GEOMETRY,
        "few_shot": RUN_FEW_SHOT, "distractors": RUN_DISTRACTOR_SCALING, "metadata_slices": RUN_METADATA_SLICES,
        "membership_hashes": memberships,
        "claim_scope": "closed-set retrieval; no inference, rejection calibration, or split rebuilding",
        "metadata_mappings": {"fallback": DATASET_CATEGORY_FALLBACK, "aliases": CATEGORY_ALIASES,
                              "columns": COLUMN_ALIASES},
    }
    protocol["protocol_id"] = digest(protocol)
    return dict(root0=root0, root1=root1, embroot=embroot, m0=m0, m1=m1, seeds=seeds,
                datasets=datasets, models=models, canonical=canon, splits=splits,
                protocol=protocol, provenance=provenance)

def load_model(inputs, model):
    result, source = {}, {}
    dimensions, checkpoints = set(), set()
    for ds in inputs["datasets"]:
        path = inputs["embroot"] / model / ds / "embeddings.npz"
        with np.load(path, allow_pickle=False) as a:
            needed = {"embeddings", "image_ids", "identities", "dataset", "metadata_json"}
            if not needed <= set(a.files): raise ValueError(f"Missing NPZ keys in {path}: {needed - set(a.files)}")
            ids, labels = a["image_ids"].astype(str), a["identities"].astype(str)
            z = normalized(a["embeddings"])
            meta = json.loads(str(a["metadata_json"].item()))
            if set(a["dataset"].astype(str).ravel()) != {ds}: raise ValueError("NPZ dataset mismatch")
        f = inputs["canonical"][ds]
        if len(z) != len(ids) or len(labels) != len(ids) or len(set(ids)) != len(ids):
            raise ValueError("Embedding rows/image IDs/identity labels are misaligned")
        if set(ids) != set(f.image_id): raise ValueError(f"Cache/canonical membership differs: {model}/{ds}")
        order = pd.Series(np.arange(len(ids)), index=ids).loc[f.image_id].to_numpy()
        if not np.array_equal(labels[order], f.identity.to_numpy()): raise ValueError("Embedding identity labels disagree")
        if meta.get("model") and meta["model"] != model: raise ValueError("Cache model provenance differs from folder name")
        if meta.get("dataset") and meta["dataset"] != ds: raise ValueError("Cache metadata dataset mismatch")
        expected_hash = inputs["m0"].get("dataset_manifest_hashes", {}).get(ds)
        if expected_hash and meta.get("dataset_manifest_sha256") != expected_hash:
            raise ValueError("Embedding cache was extracted from a different canonical manifest")
        if meta.get("image_order_sha256"):
            observed = hashlib.sha256("\n".join(ids).encode()).hexdigest()
            if observed != meta["image_order_sha256"]: raise ValueError("NPZ image-order hash differs")
        result[ds] = (f.reset_index(drop=True), z[order])
        dimensions.add(z.shape[1])
        if meta.get("checkpoint_sha256"): checkpoints.add(meta["checkpoint_sha256"])
        source[ds] = dict(cache_sha256=file_sha(path), path=str(path), metadata=meta, rows=len(z), dimension=z.shape[1])
    if len(dimensions) != 1 or len(checkpoints) > 1: raise ValueError("Datasets use different model dimensions/checkpoints")
    return result, source

def split_arrays(inputs, cache, ds, seed):
    if ds == "pooled_external":
        frames, vectors = [], []
        for name in inputs["datasets"]:
            f, z = split_arrays(inputs, cache, name, seed)
            frames.append(f); vectors.append(z)
        return pd.concat(frames, ignore_index=True), np.concatenate(vectors)
    base, z = cache[ds]
    f = inputs["splits"][(ds, seed)].reset_index(drop=True)
    index = pd.Series(np.arange(len(base)), index=base.key).loc[f.key].to_numpy()
    return f, z[index]

def split_views(frame, z):
    g, q = assert_frozen_split(frame)
    position = pd.Series(np.arange(len(frame)), index=frame.key)
    return q, z[position.loc[q.key].to_numpy()], g, z[position.loc[g.key].to_numpy()]

### Exact closed-set retrieval metrics and conditional analyses

All primary gallery images are ranked exactly. Scores and sorting are computed in bounded query blocks. Metadata-restricted retrieval retains the frozen role membership and only filters eligible gallery matches.

In [ ]:
def ranked_values(scores, positives, valid_mask=None):
    valid = np.ones(len(scores), bool) if valid_mask is None else np.asarray(valid_mask, bool)
    ix = np.flatnonzero(valid)
    if not np.any(positives[ix]): return None, np.empty(0, dtype=int)
    order = ix[np.argsort(-scores[ix], kind="stable")]
    relevant_ranks = np.flatnonzero(positives[order]) + 1
    ap = np.mean(np.arange(1, len(relevant_ranks) + 1) / relevant_ranks)
    first, last = int(relevant_ranks[0]), int(relevant_ranks[-1])
    values = dict(AP=float(ap), INP=float(len(relevant_ranks) / last), first_rank=first,
                  n_positive=len(relevant_ranks), n_gallery=len(ix), last_positive_rank=last)
    values.update({f"rank{k}": float(first <= k) for k in CONFIG["ranks"]})
    return values, order

def retrieval(q, qz, g, gz, condition="all", top_k=0):
    rows, tops = [], []
    gy, go, gkey, gc = (g[col].to_numpy() for col in ["identity_key", "observation_key", "key", "category"])
    for start in range(0, len(q), CONFIG["similarity_block"]):
        scores = np.clip(qz[start:start + CONFIG["similarity_block"]] @ gz.T, -1, 1)
        for off, score in enumerate(scores):
            a = q.iloc[start + off]
            allowed = (gkey != a.key) & (go != a.observation_key)
            if condition == "within_category": allowed &= (gc == a.category) & (a.category != "")
            elif condition in {"cross_camera", "cross_session"}:
                col = condition.split("_")[1]; values = g[col].to_numpy()
                allowed &= (values != "") & (a[col] != "") & (values != a[col])
            elif condition in {"same_side", "opposite_side"}:
                values = g.side.to_numpy()
                allowed &= (values != "") & (a.side != "")
                allowed &= values == a.side if condition == "same_side" else values != a.side
            elif condition != "all": raise ValueError(condition)
            positive = gy == a.identity_key
            values, order = ranked_values(score, positive, allowed)
            row = dict(query_id=a.key, image_id=a.image_id, identity=a.identity_key, category=a.category,
                       observation=a.observation_key, valid=values is not None, condition=condition)
            if values is None:
                row.update({m: np.nan for m in METRIC_COLUMNS})
                row.update(first_rank=np.nan, n_positive=0, n_gallery=int(allowed.sum()),
                           exclusion_reason="no_valid_gallery_positive_after_condition")
            else:
                row.update(values); row["exclusion_reason"] = ""
                pos = score[positive & allowed]
                neg = allowed & ~positive
                sib = neg & (gc == a.category) & (a.category != "")
                row.update(positive_mean=float(pos.mean()), positive_min=float(pos.min()),
                           best_negative=float(score[neg].max()) if neg.any() else np.nan,
                           best_sibling_negative=float(score[sib].max()) if sib.any() else np.nan,
                           hard_margin_all=float(pos.mean() - score[neg].max()) if neg.any() else np.nan,
                           hard_margin_sibling=float(pos.mean() - score[sib].max()) if sib.any() else np.nan)
                for rank, j in enumerate(order[:top_k], 1):
                    tops.append(dict(query_id=a.key, query_identity=a.identity_key, gallery_id=g.iloc[j].key,
                                     gallery_identity=gy[j], rank=rank, cosine=float(score[j]), correct=bool(positive[j]),
                                     query_path=a.get("path", ""), gallery_path=g.iloc[j].get("path", "")))
            rows.append(row)
    return pd.DataFrame(rows), pd.DataFrame(tops)

def retrieval_summary(perq):
    valid = perq.loc[perq.valid.astype(bool)]
    out = dict(n_queries=len(perq), n_valid_queries=len(valid), n_excluded_queries=len(perq) - len(valid),
               n_query_identities=perq.identity.nunique(), n_valid_identities=valid.identity.nunique(),
               query_coverage=len(valid)/len(perq) if len(perq) else np.nan)
    for col in METRIC_COLUMNS:
        name = {"AP": "mAP", "INP": "mINP"}.get(col, col)
        out[name] = float(valid[col].mean()) if len(valid) else np.nan
        out[f"identity_{name}"] = float(valid.groupby("identity")[col].mean().mean()) if len(valid) else np.nan
        out[f"observation_{name}"] = float(valid.groupby("observation")[col].mean().mean()) if len(valid) else np.nan
    for scope in ("all", "sibling"):
        col = f"hard_margin_{scope}"
        if col in valid:
            selected = valid.loc[pd.to_numeric(valid[col], errors="coerce").notna()]
            out[f"{col}_identity_mean"] = float(selected.groupby("identity")[col].mean().mean()) if len(selected) else np.nan
            out[f"{col}_n_queries"] = len(selected)
            if len(selected):
                weights=identity_anchor_weights(selected)
                values=selected[col].to_numpy(float)
                out[f"{col}_identity_median"]=weighted_quantile(values,weights,.5)
                out[f"{col}_identity_p10"]=weighted_quantile(values,weights,.1)
                out[f"{col}_identity_violation_rate"]=float(np.average(values<=0,weights=weights))
            else:
                for stat in ["median","p10","violation_rate"]:out[f"{col}_identity_{stat}"]=np.nan
    return out

def cmc_rows(perq):
    valid = perq.loc[perq.valid.astype(bool)]
    rows = []
    for rank in range(1, CONFIG["cmc_max_rank"] + 1):
        h = valid.assign(hit=(valid.first_rank <= rank).astype(float))
        rows.append(dict(rank=rank, success=float(h.hit.mean()) if len(h) else np.nan,
                         identity_success=float(h.groupby("identity").hit.mean().mean()) if len(h) else np.nan))
    return rows

### Matched pair sampling, verification ROC, and similarity distributions

Sampling depends on dataset, split seed, and image/identity/observation metadata, never model name or similarity scores. Pair memberships are fingerprinted. Negative sampling rotates through identities and takes distinct observations.

In [ ]:
def balanced_indices(frame, per_identity, maximum, seed, distinct_observations=False):
    rng = np.random.default_rng(seed)
    queues = {}
    for identity, group in frame.sort_values("key").groupby("identity_key", sort=True):
        ordered = rng.permutation(group.index.to_numpy()).tolist()
        if distinct_observations:
            seen, chosen = set(), []
            for i in ordered:
                observation = frame.loc[i, "observation_key"]
                if not observation: continue
                if observation not in seen: chosen.append(i); seen.add(observation)
            ordered = chosen
        queues[identity] = ordered[:per_identity]
    identities = list(queues)
    order = rng.permutation(len(identities))
    chosen = []
    for level in range(per_identity):
        for j in order:
            values = queues[identities[j]]
            if level < len(values): chosen.append(values[level])
            if len(chosen) >= maximum: return np.asarray(chosen, int)
    return np.asarray(chosen, int)

def candidate_sample(frame, candidates, maximum, seed):
    if not len(candidates): return np.empty(0, dtype=int)
    # Round-robin across identities, then observations; independent of model scores.
    f = frame.loc[candidates].copy()
    return balanced_indices(f, maximum, maximum, seed, distinct_observations=True)

def identity_anchor_weights(frame):
    if frame.empty: return np.empty(0)
    n_ids = frame.identity.nunique()
    n_queries = frame.groupby("identity").query_id.transform("nunique").to_numpy(float)
    n_pairs = frame.groupby(["identity", "query_id"]).query_id.transform("size").to_numpy(float)
    return 1.0 / (n_ids * n_queries * n_pairs)

def weighted_quantile(values, weights, probability):
    v, w = np.asarray(values, float), np.asarray(weights, float)
    keep = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if not keep.any(): return np.nan
    order = np.argsort(v[keep], kind="stable")
    v, w = v[keep][order], w[keep][order]
    ix = min(np.searchsorted(np.cumsum(w)/w.sum(), probability), len(v) - 1)
    return float(v[ix])

def verification(pairs, negative_kind):
    pos = pairs.loc[pairs.pair_kind.eq("positive")].copy()
    neg = pairs.loc[pairs.pair_kind.eq(negative_kind)].copy()
    anchors = set(pos.query_id) & set(neg.query_id)
    pos, neg = pos.loc[pos.query_id.isin(anchors)], neg.loc[neg.query_id.isin(anchors)]
    record = dict(negative_kind=negative_kind, n_positive_pairs=len(pos), n_negative_pairs=len(neg),
                  n_anchor_identities=pos.identity.nunique(), n_anchors=len(anchors), AUROC=np.nan,
                  EER_interpolated=np.nan, status="insufficient_pairs")
    for far in CONFIG["far_targets"]:
        record.update({f"TAR_at_FAR_{far:g}": np.nan, f"achieved_FAR_{far:g}": np.nan,
                       f"threshold_at_FAR_{far:g}": np.nan})
    if pos.empty or neg.empty: return record, pd.DataFrame()
    wp, wn = identity_anchor_weights(pos), identity_anchor_weights(neg)
    truth = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    scores = np.r_[pos.cosine.to_numpy(float), neg.cosine.to_numpy(float)]
    weights = np.r_[wp, wn]
    fpr, tpr, thresholds = roc_curve(truth, scores, sample_weight=weights, drop_intermediate=False)
    record["AUROC"] = float(roc_auc_score(truth, scores, sample_weight=weights))
    diff = fpr + tpr - 1
    j = int(np.flatnonzero(diff >= 0)[0])
    if j == 0 or diff[j] == 0: eer = fpr[j]
    else:
        mix = -diff[j-1] / (diff[j] - diff[j-1])
        eer = fpr[j-1] + mix * (fpr[j] - fpr[j-1])
    record["EER_interpolated"] = float(eer)
    record["status"] = "descriptive_test_ROC; EER_interpolates; TAR_uses_attainable_threshold"
    record["minimum_negative_pair_weight"] = float(wn.min() / wn.sum())
    for far in CONFIG["far_targets"]:
        allowed = np.flatnonzero(fpr <= far + 1e-12)
        j = int(allowed[np.argmax(tpr[allowed])])
        record.update({f"TAR_at_FAR_{far:g}": float(tpr[j]), f"achieved_FAR_{far:g}": float(fpr[j]),
                       f"threshold_at_FAR_{far:g}": float(thresholds[j])})
    return record, pd.DataFrame({"FAR": fpr, "TAR": tpr, "threshold": thresholds})

def pair_analysis(q, qz, g, gz, seed):
    anchors = balanced_indices(q, CONFIG["pair_queries_per_identity"], CONFIG["pair_max_queries"], seed,
                               distinct_observations=True)
    gy, gc, go, gkey = [g[c].to_numpy() for c in ["identity_key", "category", "observation_key", "key"]]
    rows = []
    for qi in anchors:
        a = q.iloc[qi]
        score = np.clip(gz @ qz[qi], -1, 1)
        valid = (gkey != a.key) & (go != a.observation_key)
        positive = valid & (gy == a.identity_key)
        negative = valid & (gy != a.identity_key)
        groups = {"positive": positive, "negative_all": negative,
                  "negative_sibling": negative & (gc == a.category) & (a.category != ""),
                  "negative_cross_category": negative & (gc != a.category) & (gc != "") & (a.category != "")}
        for kind, mask in groups.items():
            limit = CONFIG["positive_pairs_per_query"] if kind == "positive" else CONFIG["negative_pairs_per_query"]
            selected = candidate_sample(g, np.flatnonzero(mask), limit, seed_for(seed, a.key, kind))
            for j in selected:
                rows.append(dict(query_id=a.key, gallery_id=gkey[j], identity=a.identity_key,
                                 gallery_identity=gy[j], category=a.category, pair_kind=kind, cosine=float(score[j])))
    pairs = pd.DataFrame(rows, columns=["query_id", "gallery_id", "identity", "gallery_identity", "category", "pair_kind", "cosine"])
    fingerprints = {"pairs": frame_hash(pairs, ["query_id", "gallery_id", "pair_kind"]),
                    "pair_queries": digest(q.loc[anchors, "key"].tolist())}
    stats, histograms, rocs, verified = [], [], [], []
    for kind, part in pairs.groupby("pair_kind", sort=True):
        w = identity_anchor_weights(part)
        mean = float(np.average(part.cosine, weights=w))
        variance = float(np.average((part.cosine-mean)**2, weights=w))
        row = dict(pair_kind=kind, n_pairs=len(part), n_anchors=part.query_id.nunique(), mean_cosine=mean,
                   std_cosine=np.sqrt(variance), p10=weighted_quantile(part.cosine,w,.1),
                   median=weighted_quantile(part.cosine,w,.5), p90=weighted_quantile(part.cosine,w,.9))
        if kind == "positive": row["alignment_squared_l2"] = 2 - 2*mean
        stats.append(row)
        edges = np.linspace(-1,1,61)
        mass, _ = np.histogram(part.cosine, bins=edges, weights=w)
        counts, _ = np.histogram(part.cosine, bins=edges)
        for l,r,p,n in zip(edges[:-1],edges[1:],mass,counts):
            histograms.append(dict(pair_kind=kind, bin_left=l, bin_right=r, probability_mass=float(p), n_pairs=int(n)))
    for kind in ["negative_all", "negative_sibling", "negative_cross_category"]:
        r, curve = verification(pairs, kind); verified.append(r)
        if len(curve): rocs.append(curve.assign(negative_kind=kind))
        p = pairs.loc[pairs.pair_kind.eq("positive")]
        n = pairs.loc[pairs.pair_kind.eq(kind)]
        shared = set(p.query_id) & set(n.query_id)
        p, n = p.loc[p.query_id.isin(shared)], n.loc[n.query_id.isin(shared)]
        r["matched_positive_negative_gap"] = np.nan; r["d_prime_descriptive"] = np.nan
        if len(p) and len(n):
            wp, wn = identity_anchor_weights(p), identity_anchor_weights(n)
            mp, mn = np.average(p.cosine,weights=wp), np.average(n.cosine,weights=wn)
            vp = np.average((p.cosine-mp)**2,weights=wp); vn = np.average((n.cosine-mn)**2,weights=wn)
            r["matched_positive_negative_gap"] = float(mp-mn)
            r["d_prime_descriptive"] = float((mp-mn)/np.sqrt((vp+vn)/2)) if vp+vn>0 else np.nan
    return pairs, pd.DataFrame(stats), pd.DataFrame(histograms), pd.DataFrame(verified), \
        pd.concat(rocs, ignore_index=True) if rocs else pd.DataFrame(), fingerprints

### Observation-aware geometry and qualitative projections

The sample membership is exported with each result. Category silhouette on pooled data may reflect dataset/domain cues when category and source coincide. PCA/t-SNE are optional descriptive plots; no projection is fed back into retrieval or metric computation.

In [ ]:
def identity_fisher_ratio(z, labels):
    # Identity-balanced trace scatter, Euclidean on normalized embeddings.
    means, within = [], []
    for label in sorted(set(labels)):
        x = z[labels == label].astype(np.float64)
        mu = x.mean(axis=0)
        means.append(mu); within.append(float(np.mean(np.sum((x-mu)**2,axis=1))))
    means = np.asarray(means)
    sw = float(np.mean(within))
    sb = float(np.mean(np.sum((means-means.mean(axis=0))**2,axis=1))) if len(means)>1 else np.nan
    return sb/sw if sw>1e-15 else np.nan, sw, sb

def geometry_analysis(frame, z, seed):
    selected = balanced_indices(frame, CONFIG["geometry_per_identity"], CONFIG["geometry_max_samples"], seed,
                                distinct_observations=True)
    f = frame.loc[selected].reset_index(drop=True); x = z[selected].astype(np.float64)
    n, dim = x.shape
    record = dict(n_samples=n, n_identities=f.identity_key.nunique(), dimension=dim,
                  category_coverage=float(f.category.ne("").mean()) if n else np.nan,
                  membership_sha256=digest(f.key.tolist()), status="ok")
    if n<3:
        record["status"]="insufficient_observations"
        return record, f, pd.DataFrame(), selected
    gram = np.clip(x @ x.T,-1,1); distance=np.maximum(1-gram,0); np.fill_diagonal(distance,0)
    centered_gram = gram - gram.mean(axis=0)[None,:] - gram.mean(axis=1)[:,None] + gram.mean()
    eigen = np.linalg.eigvalsh(centered_gram/(n-1))[::-1]
    eigen = np.maximum(eigen,0)
    maximum = min(n-1,dim)
    eigen = eigen[:maximum]
    tolerance = max(float(eigen.max()),1.0)*1e-12
    positive = eigen[eigen>tolerance]
    if len(positive):
        p = positive/positive.sum()
        er = float(np.exp(-np.sum(p*np.log(p))))
        record.update(effective_rank_covariance=er, effective_rank_fraction=er/maximum,
                      participation_ratio=float(1/np.sum(p*p)), top_pc_variance_fraction=float(p[0]))
    else:
        record.update(effective_rank_covariance=0.0, effective_rank_fraction=0.0,
                      participation_ratio=0.0, top_pc_variance_fraction=np.nan)
    spectrum = pd.DataFrame({"component":np.arange(1,len(eigen)+1),"variance":eigen,
                             "variance_fraction":eigen/eigen.sum() if eigen.sum()>0 else np.nan})
    triangle = np.triu_indices(n,1)
    exponent = -2*CONFIG["uniformity_t"]*distance[triangle]
    record["uniformity_log_mean_exp"] = float(logsumexp(exponent)-np.log(len(exponent)))
    record["uniformity_t"] = CONFIG["uniformity_t"]
    labels=f.identity_key.to_numpy(); categories=f.category.to_numpy()
    f["identity_silhouette"] = np.nan; f["within_category_silhouette"] = np.nan
    counts = Counter(labels)
    if 2<=len(counts)<n:
        scores=silhouette_samples(distance,labels,metric="precomputed")
        eligible=np.asarray([counts[y]>=2 for y in labels])
        f.loc[eligible,"identity_silhouette"]=scores[eligible]
    record["identity_silhouette"] = float(f.groupby("identity_key").identity_silhouette.mean().mean())
    record["silhouette_eligible_samples"] = int(f.identity_silhouette.notna().sum())
    ratios, within_distances, between_distances, uniforms = [], [], [], []
    for category in sorted(set(categories)-{""}):
        ix=np.flatnonzero(categories==category); lab=labels[ix]
        if 2<=len(set(lab))<len(ix):
            sc=silhouette_samples(distance[np.ix_(ix,ix)],lab,metric="precomputed")
            count=Counter(lab); eligible=np.asarray([count[y]>=2 for y in lab])
            f.loc[ix[eligible],"within_category_silhouette"]=sc[eligible]
        if len(set(lab))>=2:
            ratio,sw,sb=identity_fisher_ratio(x[ix],lab)
            ratios.append(ratio);within_distances.append(sw);between_distances.append(sb)
        if len(ix)>1:
            d=distance[np.ix_(ix,ix)][np.triu_indices(len(ix),1)]
            uniforms.append(float(logsumexp(-2*CONFIG["uniformity_t"]*d)-np.log(len(d))))
    record["within_category_identity_silhouette"] = float(f.loc[f.category.ne("")].groupby(
        ["category","identity_key"]).within_category_silhouette.mean().groupby(level=0).mean().mean())
    record["within_category_fisher_trace_ratio"] = float(pd.Series(ratios,dtype=float).mean())
    record["within_category_uniformity"] = float(pd.Series(uniforms,dtype=float).mean())
    record["fisher_trace_ratio"],record["within_identity_scatter"],record["between_identity_scatter"] = identity_fisher_ratio(x,labels)
    # Explicit pairwise within-ID distance, with equal identity weights.
    within=[]
    for identity in sorted(set(labels)):
        ix=np.flatnonzero(labels==identity)
        if len(ix)>1: within.append(float(distance[np.ix_(ix,ix)][np.triu_indices(len(ix),1)].mean()))
    record["within_identity_cosine_distance"] = float(pd.Series(within,dtype=float).mean())
    known=np.flatnonzero(categories!=""); record["category_silhouette"]=np.nan
    if len(known) and 2<=len(set(categories[known]))<len(known):
        scores=silhouette_samples(distance[np.ix_(known,known)],categories[known],metric="precomputed")
        c=Counter(categories[known]); good=np.asarray([c[t]>=2 for t in categories[known]])
        tmp=pd.DataFrame({"category":categories[known][good],"score":scores[good]})
        record["category_silhouette"]=float(tmp.groupby("category").score.mean().mean())
    return record,f,spectrum,selected

def projection_plots(f, z, folder, seed):
    if len(f)<3:return
    projections={}
    if RUN_PCA_PLOTS:
        projections["PCA"]=PCA(n_components=2,svd_solver="randomized",random_state=seed).fit_transform(z)
    if RUN_TSNE_PLOTS:
        from sklearn.manifold import TSNE
        projections["tSNE"]=TSNE(n_components=2,metric="cosine",init="random",learning_rate="auto",
                                 perplexity=min(30,max(2,(len(f)-1)/3)),random_state=seed).fit_transform(z)
    selected_ids=set(sorted(f.identity_key.unique())[:20])
    for name,xy in projections.items():
        save_table(folder/f"{name}_coordinates.csv",f[["key","identity_key","category"]].assign(x=xy[:,0],y=xy[:,1]))
        fig,axes=plt.subplots(1,2,figsize=(12,4.5))
        for j,identity in enumerate(sorted(selected_ids)):
            ix=f.identity_key.eq(identity).to_numpy()
            axes[0].scatter(xy[ix,0],xy[ix,1],s=14,color=plt.get_cmap("tab20")(j),alpha=.8)
        axes[0].set_title(f"{name}: fixed subset of up to 20 identities")
        for category,part in f.groupby("category",sort=True):
            ix=part.index.to_numpy();axes[1].scatter(xy[ix,0],xy[ix,1],s=10,label=category or "missing",alpha=.6)
        axes[1].set_title(f"{name}: animal categories (descriptive)")
        if f.category.nunique()<=15:axes[1].legend(fontsize=8)
        fig.suptitle("Qualitative projection only; numerical metrics use original embeddings")
        save_figure(fig,folder/f"{name}.png")

### Limited-gallery and controlled distractor experiments

Few-shot means one gallery image from each of 1, 2, or 5 distinct frozen gallery observations. It exports both per-k eligible identities and a shared max-k cohort. Distractor sets are nested and model-independent. Sibling curves start from the query's positive gallery images; cross-category curves retain all same-category gallery images before adding other-category identities. Common-query curves are exported alongside variable-coverage curves.

In [ ]:
def few_shot_analysis(q,qz,g,gz,seed):
    rows, details, fingerprints = [], [], {}
    grouped={}
    for identity,part in g.groupby("identity_key",sort=True):
        groups=[]
        rng=np.random.default_rng(seed_for(seed,identity,"few-shot"))
        for observation,obs in part.sort_values("key").groupby("observation_key",sort=True):
            groups.append(int(rng.choice(obs.index.to_numpy())))
        grouped[identity]=rng.permutation(groups).tolist()
    max_k=max(CONFIG["few_shot_observations"])
    common={y for y,indices in grouped.items() if len(indices)>=max_k}
    for cohort in ("eligible_per_k","common_max_k_cohort"):
        for k in CONFIG["few_shot_observations"]:
            eligible={y for y,ix in grouped.items() if len(ix)>=k}
            if cohort=="common_max_k_cohort":eligible &= common
            gi=sorted([j for identity in sorted(eligible) for j in grouped[identity][:k]],key=lambda j:g.iloc[j].key)
            qi=np.flatnonzero(q.identity_key.isin(eligible))
            key=f"{cohort}/{k}"
            fingerprints[key]=digest(dict(gallery=g.iloc[gi].key.tolist(),queries=q.iloc[qi].key.tolist()))
            if not len(qi) or not len(gi):
                rows.append(dict(cohort=cohort,gallery_observations=k,status="no_eligible_identities",n_original_queries=len(q),n_queries=0))
                continue
            p,_=retrieval(q.iloc[qi].reset_index(drop=True),qz[qi],g.iloc[gi].reset_index(drop=True),gz[gi])
            details.append(p.assign(cohort=cohort,gallery_observations=k))
            rows.append(dict(cohort=cohort,gallery_observations=k,status="ok",n_original_queries=len(q),
                             retained_query_fraction=len(qi)/len(q),**retrieval_summary(p)))
    return pd.DataFrame(rows),pd.concat(details,ignore_index=True) if details else pd.DataFrame(),fingerprints

def distractor_analysis(q,qz,g,gz,seed):
    anchors=balanced_indices(q,CONFIG["distractor_queries_per_identity"],CONFIG["distractor_max_queries"],seed,True)
    gy,gc=g.identity_key.to_numpy(),g.category.to_numpy()
    keyhash=[];details=[]
    for qi in anchors:
        a=q.iloc[qi]
        if not a.category:continue
        score=np.clip(gz @ qz[qi],-1,1)
        valid=(g.key.to_numpy()!=a.key)&(g.observation_key.to_numpy()!=a.observation_key)
        positive=valid&(gy==a.identity_key)
        if not positive.any():continue
        sibling=valid&(gy!=a.identity_key)&(gc==a.category)
        cross=valid&(gc!="")&(gc!=a.category)
        for scope,mask in (("sibling",sibling),("cross_category",cross)):
            available=sorted(set(gy[mask]))
            rng=np.random.default_rng(seed_for(seed,a.key,scope))
            order=[available[i] for i in rng.permutation(len(available))]
            # Cross-category experiment retains ALL same-category gallery images as its base.
            base=positive if scope=="sibling" else positive|sibling
            for count in CONFIG["distractor_counts"]:
                if count>len(order):continue
                selected=order[:count]
                allowed=base|(valid&np.isin(gy,selected))
                values,_=ranked_values(score,positive,allowed)
                keyhash.append([a.key,scope,count,g.loc[allowed,"key"].tolist()])
                details.append(dict(query_id=a.key,identity=a.identity_key,observation=a.observation_key,
                                    category=a.category,valid=True,scope=scope,n_distractor_identities=count,
                                    n_available_distractor_identities=len(order),**values))
    details=pd.DataFrame(details)
    rows=[]
    if len(details):
        for (scope,count),part in details.groupby(["scope","n_distractor_identities"],sort=True):
            rows.append(dict(scope=scope,n_distractor_identities=count,cohort="eligible_per_count",
                             **retrieval_summary(part)))
        # A second curve uses the common query set across all available tested counts.
        for scope,part in details.groupby("scope",sort=True):
            counts=sorted(part.n_distractor_identities.unique())
            eligible=set.intersection(*(set(part.loc[part.n_distractor_identities.eq(c),"query_id"]) for c in counts))
            for count in counts:
                subset=part.loc[part.n_distractor_identities.eq(count)&part.query_id.isin(eligible)]
                if len(subset):rows.append(dict(scope=scope,n_distractor_identities=count,
                                               cohort="common_queries_across_counts",**retrieval_summary(subset)))
    return pd.DataFrame(rows),details,digest(keyhash)

### Summaries and fixed-gallery identity-block bootstrap

Both query-weighted and identity-balanced retrieval estimates receive conditional bootstrap intervals. Paired comparisons require identical query labels and validity masks. Repeated split seeds are averaged within each resampled identity structure, not treated as independent training runs.

In [ ]:
def metric_summary(results, keys, expected_splits=None):
    excluded={*keys,"n_queries","n_valid_queries","n_excluded_queries","n_query_identities","n_valid_identities",
              "query_coverage","split_seed","dimension"}
    numeric=[c for c in results.columns if c not in excluded and pd.api.types.is_numeric_dtype(results[c])
             and not c.startswith("n_") and not c.endswith("_n_queries")]
    rows=[]
    for group_key,part in results.groupby(keys,dropna=False,sort=True):
        group_key=group_key if isinstance(group_key,tuple) else (group_key,)
        for metric in numeric:
            values=pd.to_numeric(part[metric],errors="coerce").replace([np.inf,-np.inf],np.nan)
            expected=len(part) if expected_splits is None else expected_splits
            rows.append(dict(zip(keys,group_key),metric=metric,mean=float(values.mean()),
                             split_sd=float(values.std(ddof=1)),n_valid_splits=int(values.notna().sum()),
                             n_expected_splits=expected,complete=bool(values.notna().sum()==expected)))
    return pd.DataFrame(rows)

def dataset_macro(summary,datasets,condition="all"):
    rows=[]
    for metric,part in summary.loc[summary.condition.eq(condition)&summary.dataset.isin(datasets)].groupby("metric"):
        complete=set(part.dataset)==set(datasets) and part.complete.all()
        rows.append(dict(condition=condition,metric=metric,mean=float(part["mean"].mean()) if complete else np.nan,
                         available_dataset_mean=float(part.loc[part.complete,"mean"].mean()),
                         n_complete_datasets=int(part.complete.sum()),n_expected_datasets=len(datasets),complete=complete))
    return pd.DataFrame(rows)

def bootstrap_retrieval(perq, dataset_names, condition="all", other=None):
    """Dataset-stratified query-identity bootstrap, with the same identity draw across seeds.

    Gallery and trained checkpoint(s) are fixed. Splits are averaged, never resampled
    as if independent datasets. For paired differences all query memberships must match.
    """
    a=perq.loc[perq.condition.eq(condition)&perq.dataset.isin(dataset_names)].copy()
    if other is not None:
        b=other.loc[other.condition.eq(condition)&other.dataset.isin(dataset_names)].copy()
        keys=["dataset","split_seed","query_id"]
        a=a.sort_values(keys).reset_index(drop=True);b=b.sort_values(keys).reset_index(drop=True)
        if a.duplicated(keys).any() or b.duplicated(keys).any():raise ValueError("Duplicate queries in paired comparison")
        if not a[keys].equals(b[keys]) or not np.array_equal(a.valid.to_numpy(),b.valid.to_numpy()) \
           or not np.array_equal(a.identity.to_numpy(),b.identity.to_numpy()):
            raise ValueError("Paired queries/identity/validity differ; comparison refused")
        for col in METRIC_COLUMNS:a[col]=pd.to_numeric(a[col],errors="coerce")-pd.to_numeric(b[col],errors="coerce")
    a=a.loc[a.valid.astype(bool)]
    B=int(CONFIG["bootstrap_replicates"]);alpha=(1-CONFIG["confidence_level"])/2
    records=[];by_dataset={}
    for ds in dataset_names:
        f=a.loc[a.dataset.eq(ds)]
        if f.empty:continue
        ids=sorted(f.identity.unique());seeds=sorted(f.split_seed.unique())
        rng=np.random.default_rng(seed_for(CONFIG["random_seed"],condition,ds,"identity-block-bootstrap"))
        draws=rng.multinomial(len(ids),np.full(len(ids),1/len(ids)),size=B).astype(float)
        ds_reps={}
        for col in METRIC_COLUMNS:
            values=f.assign(value=pd.to_numeric(f[col],errors="coerce")).groupby(["identity","split_seed"]).value.agg(["sum","count","mean"])
            index=pd.MultiIndex.from_product([ids,seeds],names=["identity","split_seed"])
            values=values.reindex(index)
            sums=values["sum"].fillna(0).to_numpy().reshape(len(ids),len(seeds))
            counts=values["count"].fillna(0).to_numpy().reshape(len(ids),len(seeds))
            means=values["mean"].fillna(0).to_numpy().reshape(len(ids),len(seeds))
            present=(counts>0).astype(float)
            for weighting,numer,denom in (("query",sums,counts),("identity",means,present)):
                metric={"AP":"mAP","INP":"mINP"}.get(col,col)
                if weighting=="identity":metric="identity_"+metric
                point=float(np.mean(numer.sum(axis=0)/denom.sum(axis=0)))
                den=draws@denom
                with np.errstate(divide="ignore",invalid="ignore"):
                    reps=np.mean((draws@numer)/den,axis=1)
                reps[np.any(den<=0,axis=1)]=np.nan
                finite=reps[np.isfinite(reps)]
                low,high=np.quantile(finite,[alpha,1-alpha]) if len(finite) else (np.nan,np.nan)
                records.append(dict(dataset=ds,condition=condition,metric=metric,estimate=point,ci_low=low,ci_high=high,
                                    n_identities=len(ids),n_splits=len(seeds),valid_bootstrap_replicates=len(finite),
                                    bootstrap_replicates=B,confidence_level=CONFIG["confidence_level"],
                                    inference_scope="query identities; fixed gallery, checkpoint, datasets, and split seeds"))
                ds_reps[metric]=(point,reps)
        by_dataset[ds]=ds_reps
    if set(by_dataset)==set(dataset_names):
        for metric in next(iter(by_dataset.values())):
            point=np.mean([by_dataset[d][metric][0] for d in dataset_names])
            reps=np.mean(np.stack([by_dataset[d][metric][1] for d in dataset_names]),axis=0)
            finite=reps[np.isfinite(reps)]
            low,high=np.quantile(finite,[alpha,1-alpha]) if len(finite) else (np.nan,np.nan)
            records.append(dict(dataset="dataset_macro",condition=condition,metric=metric,estimate=float(point),
                                ci_low=low,ci_high=high,n_datasets=len(dataset_names),valid_bootstrap_replicates=len(finite),
                                bootstrap_replicates=B,confidence_level=CONFIG["confidence_level"],
                                inference_scope="fixed evaluated datasets; independent identity resampling within each dataset"))
    return pd.DataFrame(records)

### Plotting and inspectable output bundles

Plots are saved even when SHOW_PLOTS=False. Optional image panels use a fixed metadata-selected query sample, with no score-based example selection. Missing source images do not affect numerical evaluation.

In [ ]:
def plot_split_tables(folder, perq, cmc, hist, roc, geometry, spectrum, few, distractors):
    if len(cmc):
        fig,ax=plt.subplots(figsize=(6,4))
        for condition,part in cmc.groupby("condition"):
            ax.plot(part["rank"],part.success,label=condition)
        ax.set(xlabel="Gallery image rank",ylabel="CMC success",ylim=(0,1.02));ax.legend(fontsize=8)
        save_figure(fig,folder/"cmc.png")
    if len(hist):
        fig,ax=plt.subplots(figsize=(7,4))
        for kind,part in hist.groupby("pair_kind"):
            ax.plot((part.bin_left+part.bin_right)/2,part.probability_mass,label=kind)
        ax.set(xlabel="Cosine similarity",ylabel="Identity-balanced probability mass");ax.legend(fontsize=8)
        save_figure(fig,folder/"pair_similarity_distributions.png")
    if len(roc):
        fig,ax=plt.subplots(figsize=(6,4))
        for kind,part in roc.groupby("negative_kind"):
            ax.plot(np.maximum(part.FAR,1e-6),part.TAR,label=kind)
        ax.set(xscale="log",xlabel="False acceptance rate",ylabel="True acceptance rate",ylim=(0,1.02));ax.legend(fontsize=8)
        save_figure(fig,folder/"verification_ROC.png")
    if len(perq):
        fig,axes=plt.subplots(1,2,figsize=(10,4))
        main=perq.loc[perq.condition.eq("all")&perq.valid.astype(bool)]
        means=main.groupby("identity").AP.mean()
        axes[0].hist(means,bins=20,color="#218c8d");axes[0].set(xlabel="Identity mean AP",ylabel="Identities")
        for scope in ["all","sibling"]:
            col=f"hard_margin_{scope}"
            if col in main:
                vals=pd.to_numeric(main[col],errors="coerce").dropna()
                if len(vals):axes[1].hist(vals,bins=30,alpha=.45,label=scope)
        axes[1].axvline(0,color="black",lw=1);axes[1].set(xlabel="Mean-positive minus strongest-impostor cosine",ylabel="Queries")
        if axes[1].get_legend_handles_labels()[0]:axes[1].legend()
        save_figure(fig,folder/"identity_and_margin_distributions.png")
    if len(spectrum):
        fig,ax=plt.subplots(figsize=(6,4));ax.plot(spectrum.component,spectrum.variance_fraction.cumsum())
        ax.set(xlabel="Principal components",ylabel="Cumulative variance",ylim=(0,1.02))
        save_figure(fig,folder/"covariance_spectrum.png")
    if len(few) and "mAP" in few:
        fig,ax=plt.subplots(figsize=(6,4))
        for cohort,part in few.groupby("cohort"):
            ax.plot(part.gallery_observations,part.mAP,marker="o",label=cohort)
        ax.set(xlabel="Gallery observations per identity",ylabel="mAP",ylim=(0,1.02));ax.legend(fontsize=8)
        save_figure(fig,folder/"few_shot.png")
    if len(distractors):
        fig,ax=plt.subplots(figsize=(7,4))
        selected=distractors.loc[distractors.cohort.eq("common_queries_across_counts")]
        for scope,part in selected.groupby("scope"):
            ax.plot(part.n_distractor_identities,part.mAP,marker="o",label=scope)
        ax.set(xlabel="Added distractor identities",ylabel="mAP (common query cohort)",ylim=(0,1.02));ax.legend(fontsize=8)
        save_figure(fig,folder/"distractor_scaling.png")

def maybe_image_panels(top, folder, seed):
    if not EXPORT_IMAGE_PANELS or top.empty:return "disabled"
    from PIL import Image
    keys=sorted(top.query_id.unique());rng=np.random.default_rng(seed)
    keys=[keys[i] for i in rng.permutation(len(keys))[:4]]
    def resolve(s):
        for old,new in IMAGE_PATH_REWRITES.items():
            if str(s).startswith(old):s=new+str(s)[len(old):];break
        return Path(s)
    fig,axes=plt.subplots(len(keys),6,figsize=(13,2.5*len(keys)),squeeze=False)
    found=0
    for r,key in enumerate(keys):
        matches=top.loc[top.query_id.eq(key)].sort_values("rank").head(5)
        paths=[matches.query_path.iloc[0]]+matches.gallery_path.tolist()
        titles=["Query"]+[f"#{int(v['rank'])} {'correct' if v['correct'] else 'wrong'}" for _,v in matches.iterrows()]
        for c in range(6):
            ax=axes[r,c];ax.axis("off")
            if c>=len(paths):continue
            p=resolve(paths[c]);ax.set_title(titles[c],fontsize=9)
            if p.is_file():
                try:
                    with Image.open(p) as image:ax.imshow(image.convert("RGB"));found+=1
                except OSError:ax.text(.5,.5,"unreadable",ha="center")
            else:ax.text(.5,.5,"image unavailable",ha="center")
    if found:save_figure(fig,folder/"fixed_query_retrieval_examples.png")
    else:plt.close(fig)
    return "saved" if found else "skipped_no_source_images"

### Per-model evaluation orchestration

Each model writes a versioned result folder. Completed folders can be resumed only after output checksums are verified. A result ZIP contains CSVs, figures, protocol/provenance, and a run manifest; it contains no model weights or embedding arrays.

In [ ]:
def validate_configuration():
    if MODE not in {"evaluate","compare","both"}:raise ValueError("MODE must be evaluate, compare, or both")
    for name in ["similarity_block","pair_queries_per_identity","pair_max_queries","positive_pairs_per_query",
                 "negative_pairs_per_query","geometry_per_identity","geometry_max_samples","bootstrap_replicates"]:
        if not isinstance(CONFIG[name],int) or CONFIG[name]<1:raise ValueError(f"CONFIG[{name}] must be a positive integer")
    if not 0<CONFIG["confidence_level"]<1:raise ValueError("Invalid confidence_level")
    if any(not 0<float(f)<1 for f in CONFIG["far_targets"]):raise ValueError("Invalid FAR target")
    if not CONFIG["few_shot_observations"] or min(CONFIG["few_shot_observations"])<1:raise ValueError("Invalid few-shot counts")
    if not CONFIG["distractor_counts"] or min(CONFIG["distractor_counts"])<0:raise ValueError("Invalid distractor counts")

def verify_completed_bundle(folder, manifest, artifacts=None):
    if manifest.get("status")!="complete":raise ValueError(f"Incomplete result bundle: {folder}")
    checksums = manifest.get("output_sha256", {})
    if artifacts is not None and set(artifacts) - set(checksums):
        raise ValueError(f"Missing checksum for comparison inputs in {folder}")
    selected = checksums if artifacts is None else artifacts
    for relative in selected:
        expected = checksums[relative]
        p=Path(folder)/relative
        if not p.is_file() or file_sha(p)!=expected:raise ValueError(f"Result artifact changed or missing: {p}")

def evaluate_model(inputs,model):
    started=time.perf_counter();print(f"\nLoading {model}",flush=True)
    cache,source=load_model(inputs,model)
    model_info={"model":model,"label":model,"family":"unspecified","loss":"unspecified",
                "training_seed":None,"ablation":"unspecified",**MODEL_METADATA.get(model,{})}
    model_info["embedding_dimension"]=next(iter(cache.values()))[1].shape[1]
    signature=digest(dict(protocol=inputs["protocol"]["protocol_id"],
                          caches={d:s["cache_sha256"] for d,s in source.items()},model_info=model_info))
    folder=OUTPUT_ROOT.resolve()/"runs"/f"{safe_name(model)}__{signature[:12]}"
    manifest_path=folder/"run_manifest.json"
    if RESUME_COMPLETED and manifest_path.is_file():
        old=json.loads(manifest_path.read_text())
        if old.get("run_id")==signature and old.get("status")=="complete":
            verify_completed_bundle(folder,old)
            print(f"Verified completed run: {folder}",flush=True);return folder
    folder.mkdir(parents=True,exist_ok=True)
    write_json(manifest_path,{"status":"running","run_id":signature,"model_info":model_info})
    write_json(folder/"protocol.json",inputs["protocol"])
    write_json(folder/"model_provenance.json",{"model_info":model_info,"sources":source})
    statuses=[];retrieval_rows=[];query_tables=[];cmc_tables=[];verification_tables=[]
    geometry_rows=[];fingerprints=[];pair_stats_tables=[];few_tables=[];distractor_tables=[]
    datasets=inputs["datasets"]+(["pooled_external"] if RUN_POOLED else [])
    for ds in datasets:
        for split_seed in inputs["seeds"]:
            task_start=time.perf_counter();print(f"  {ds} / split {split_seed}",flush=True)
            f,z=split_arrays(inputs,cache,ds,split_seed)
            q,qz,g,gz=split_views(f,z)
            out=folder/"splits"/ds/f"seed_{split_seed}";out.mkdir(parents=True,exist_ok=True)
            base=dict(dataset=ds,split_seed=split_seed)
            sampling_seed=seed_for(CONFIG["random_seed"],ds,split_seed)
            fingerprints.append({**base,"analysis":"roles","hash":digest(dict(queries=q.key.tolist(),gallery=g.key.tolist()))})
            main,top=retrieval(q,qz,g,gz,top_k=CONFIG["export_top_k"])
            conditions=["all"]
            if q.category.ne("").any() and g.category.ne("").any():conditions.append("within_category")
            else:status_record(statuses,ds,split_seed,"within_category","skipped","missing category labels")
            if RUN_METADATA_SLICES:
                for condition,column in [("cross_camera","camera"),("cross_session","session"),
                                         ("same_side","side"),("opposite_side","side")]:
                    if q[column].ne("").any() and g[column].ne("").any():conditions.append(condition)
                    else:status_record(statuses,ds,split_seed,condition,"skipped",f"missing reliable {column} metadata")
            per_split=[];cmc_split=[]
            for condition in conditions:
                if condition=="all":p=main
                elif condition=="within_category" and q.category.nunique()==g.category.nunique()==1 \
                     and q.category.iloc[0]==g.category.iloc[0] and q.category.iloc[0]!="":
                    p=main.copy();p["condition"]=condition
                else:p,_=retrieval(q,qz,g,gz,condition)
                summary=retrieval_summary(p)
                retrieval_rows.append({**base,"condition":condition,**summary})
                per_split.append(p.assign(**base))
                cmc_split.extend([{**base,"condition":condition,**r} for r in cmc_rows(p)])
                status_record(statuses,ds,split_seed,condition,"ok" if summary["n_valid_queries"] else "unavailable",
                              "" if summary["n_valid_queries"] else "no positive remains after metadata condition",
                              n_queries=len(p),n_valid_queries=summary["n_valid_queries"])
            per_split=pd.concat(per_split,ignore_index=True);query_tables.append(per_split)
            cmc_frame=pd.DataFrame(cmc_split);cmc_tables.append(cmc_frame)
            save_table(out/"per_query_retrieval.csv.gz",per_split)
            save_table(out/"top_gallery_matches.csv.gz",top)
            save_table(out/"cmc.csv",cmc_frame)
            image_status=maybe_image_panels(top,out,sampling_seed)
            if EXPORT_IMAGE_PANELS:status_record(statuses,ds,split_seed,"retrieval_panels",image_status)
            hist=roc=geometry_detail=spectrum=few=distractors=pd.DataFrame()
            if RUN_PAIR_ANALYSIS:
                pairs,pstats,hist,vr,roc,hashes=pair_analysis(q,qz,g,gz,sampling_seed)
                for analysis,h in hashes.items():fingerprints.append({**base,"analysis":analysis,"hash":h})
                save_table(out/"sampled_pair_scores.csv.gz",pairs)
                save_table(out/"pair_statistics.csv",pstats)
                save_table(out/"pair_histograms.csv",hist)
                save_table(out/"verification_ROC.csv.gz",roc)
                vr["category_scope"]="all_categories"
                per_category=[]
                for category,part in pairs.loc[pairs.category.ne("")].groupby("category",sort=True):
                    r,_=verification(part,"negative_sibling");r["category_scope"]=category;per_category.append(r)
                vr=pd.concat([vr,pd.DataFrame(per_category)],ignore_index=True)
                save_table(out/"verification.csv",vr)
                verification_tables.append(vr.assign(**base));pair_stats_tables.append(pstats.assign(**base))
                status_record(statuses,ds,split_seed,"pair_analysis","ok",n_pairs=len(pairs))
                del pairs
            if RUN_GEOMETRY:
                active=f.closed_role.isin(["gallery","query_known"]).to_numpy()
                gf=f.loc[active].reset_index(drop=True);gz_union=z[active]
                geom,geometry_detail,spectrum,chosen=geometry_analysis(gf,gz_union,sampling_seed)
                geometry_rows.append({**base,**geom})
                fingerprints.append({**base,"analysis":"geometry","hash":geom["membership_sha256"]})
                save_table(out/"geometry_samples.csv",geometry_detail)
                save_table(out/"covariance_spectrum.csv",spectrum)
                if split_seed==inputs["seeds"][0]:projection_plots(geometry_detail,gz_union[chosen],out,sampling_seed)
                status_record(statuses,ds,split_seed,"geometry",geom["status"],n_samples=geom["n_samples"])
            if RUN_FEW_SHOT:
                few,fd,hashes=few_shot_analysis(q,qz,g,gz,sampling_seed)
                few_tables.append(few.assign(**base))
                save_table(out/"few_shot.csv",few);save_table(out/"few_shot_per_query.csv.gz",fd)
                for setting,h in hashes.items():fingerprints.append({**base,"analysis":"few_shot/"+setting,"hash":h})
                status_record(statuses,ds,split_seed,"few_shot","ok" if len(few) else "unavailable")
            if RUN_DISTRACTOR_SCALING:
                distractors,dd,hash_=distractor_analysis(q,qz,g,gz,sampling_seed)
                distractor_tables.append(distractors.assign(**base))
                save_table(out/"distractor_scaling.csv",distractors);save_table(out/"distractor_per_query.csv.gz",dd)
                fingerprints.append({**base,"analysis":"distractors","hash":hash_})
                status_record(statuses,ds,split_seed,"distractors","ok" if len(dd) else "unavailable",
                              "" if len(dd) else "missing categories or no eligible anchors")
            plot_split_tables(out,per_split,cmc_frame,hist,roc,geometry_detail,spectrum,few,distractors)
            status_record(statuses,ds,split_seed,"split_total","complete",seconds=round(time.perf_counter()-task_start,3))
            gc.collect()
    def concat(tables):return pd.concat([t for t in tables if len(t)],ignore_index=True) if any(len(t) for t in tables) else pd.DataFrame()
    retrieval_frame=save_table(folder/"retrieval_results.csv",retrieval_rows)
    all_queries=save_table(folder/"per_query_retrieval.csv.gz",concat(query_tables))
    save_table(folder/"cmc_curves.csv",concat(cmc_tables))
    summary=save_table(folder/"retrieval_summary.csv",metric_summary(retrieval_frame,["dataset","condition"],len(inputs["seeds"])))
    macro=save_table(folder/"dataset_macro_summary.csv",dataset_macro(summary,inputs["datasets"]))
    cis=[]
    for condition in ("all","within_category"):
        if condition in set(all_queries.condition):cis.append(bootstrap_retrieval(all_queries,inputs["datasets"],condition))
    save_table(folder/"retrieval_confidence_intervals.csv",concat(cis))
    save_table(folder/"verification_results.csv",concat(verification_tables))
    save_table(folder/"pair_statistics.csv",concat(pair_stats_tables))
    save_table(folder/"geometry_results.csv",geometry_rows)
    save_table(folder/"few_shot_results.csv",concat(few_tables))
    save_table(folder/"distractor_results.csv",concat(distractor_tables))
    save_table(folder/"analysis_status.csv",statuses)
    save_table(folder/"sampling_fingerprints.csv",fingerprints)
    report = f"""# {model_info['label']}

Completed {len(datasets)} dataset views x {len(inputs['seeds'])} frozen splits.
Main aggregate: equal weight to {', '.join(inputs['datasets'])}; pooled_external is supplementary.
All scores are fractions, not percentages. CMC and Rank-k rank gallery images.

Important files:
- retrieval_results.csv: per-split metrics with eligibility counts.
- retrieval_summary.csv / dataset_macro_summary.csv: descriptive split averages.
- retrieval_confidence_intervals.csv: identity-block bootstrap, conditional on this checkpoint and galleries.
- verification_results.csv: same/different identity verification; no novelty detection.
- geometry_results.csv: sampled original-space diagnostics, not universal representation scores.
- per_query_retrieval.csv.gz: paired comparison source, including AP, INP, ranks, and exact hard margins.
- analysis_status.csv: skipped/unsupported analyses and reasons.
- sampling_fingerprints.csv: membership checks for later comparisons.

No test-selected deployment threshold is produced. TAR@FAR is descriptive ROC analysis.
EER is linearly interpolated at the ROC crossing; TAR@FAR never splits ties or interpolates.
Pair samples weight identities, query observations, then sampled pairs equally.
Geometry uses at most one retained image per recorded observation.
Few-shot exports both eligible-per-k and a common max-k identity cohort.
Cross-category distractor scaling retains the full same-category gallery as its base.
Metadata slices require explicit reliable metadata and report lost query coverage.
Uncertainty does not include training variation; report separate training runs explicitly.
The meaningful hierarchy, positive-only control, and sub-center-count ablations need separately trained checkpoints.
This run does not establish absence of evaluation identities from backbone pretraining.

Protocol ID: {inputs['protocol']['protocol_id']}
Run ID: {signature}
"""
    (folder/"README.md").write_text(report,encoding="utf-8")
    outputs={str(p.relative_to(folder)):file_sha(p) for p in sorted(folder.rglob("*"))
             if p.is_file() and p.name!="run_manifest.json"}
    manifest={"status":"complete","version":VERSION,"run_id":signature,"model_info":model_info,
              "protocol_id":inputs["protocol"]["protocol_id"],"protocol":inputs["protocol"],
              "sources":source,"source_manifest_sha256":inputs["provenance"],"output_sha256":outputs,
              "completed_utc":datetime.now(timezone.utc).isoformat(),"duration_seconds":time.perf_counter()-started,
              "environment":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,
                             "scipy":scipy.__version__,"sklearn":sklearn.__version__,"matplotlib":matplotlib.__version__}}
    write_json(manifest_path,manifest)
    if MAKE_ZIP:
        archive=shutil.make_archive(str(folder),"zip",root_dir=folder.parent,base_dir=folder.name)
        print(f"  Portable result bundle: {archive}",flush=True)
    print(f"  Complete in {(time.perf_counter()-started)/60:.1f} min: {folder}",flush=True)
    display(macro.loc[macro.metric.isin(["mAP","rank1","identity_mAP","identity_rank1","mINP"])])
    del cache;gc.collect()
    return folder

def run_evaluation():
    validate_configuration();inputs=prepare_inputs()
    print("Stage 00:",inputs["root0"]);print("Stage 01:",inputs["root1"])
    print("Models:",inputs["models"],"Datasets:",inputs["datasets"],"Frozen seeds:",inputs["seeds"])
    print("Protocol:",inputs["protocol"]["protocol_id"],"\nCPU evaluation: no PyTorch, model loading, or network access.")
    with threadpool_limits(limits=CONFIG["numeric_threads"]):
        completed=[evaluate_model(inputs,model) for model in inputs["models"]]
    return completed

### Compare independent completed runs without loading their embeddings

Comparison reads completed result folders or explicitly supplied ZIPs, verifies artifacts, and refuses different protocols or sampling fingerprints. It reports all pairs of distinct runs. Family/loss/ablation/training-seed labels are recorded only when supplied explicitly; cross-backbone differences do not isolate a loss effect.

In [ ]:
def comparison_manifests(roots):
    import zipfile
    discovered=[]
    for value in roots:
        root=Path(value).expanduser().resolve()
        if root.is_file() and root.suffix.lower()==".zip":
            target=OUTPUT_ROOT.resolve()/"comparison_inputs"/file_sha(root)[:16]
            with zipfile.ZipFile(root) as archive:
                for member in archive.infolist():
                    p=Path(member.filename)
                    if p.is_absolute() or ".." in p.parts or ((member.external_attr>>16)&0o170000)==0o120000:
                        raise ValueError("Unsafe path or symbolic link in result archive")
                archive.extractall(target)
            root=target
        if root.is_file() and root.name=="run_manifest.json":discovered.append(root)
        elif root.is_dir():discovered.extend(sorted(root.rglob("run_manifest.json")))
        else:raise FileNotFoundError(f"No result directory/archive at {root}")
    bundles={}
    for p in sorted(set(discovered)):
        m=json.loads(p.read_text())
        if m.get("status")!="complete":continue
        required = ("per_query_retrieval.csv.gz", "retrieval_summary.csv", "dataset_macro_summary.csv",
                    "cmc_curves.csv", "sampling_fingerprints.csv")
        optional = ("verification_results.csv", "geometry_results.csv", "few_shot_results.csv", "distractor_results.csv")
        available = tuple(name for name in optional if (p.parent / name).is_file())
        verify_completed_bundle(p.parent, m, artifacts=required + available)
        rid=m["run_id"]
        if rid in bundles:
            if bundles[rid][1]["output_sha256"]!=m["output_sha256"]:
                raise ValueError(f"Duplicate run ID has conflicting output files: {rid}")
            continue
        bundles[rid]=(p.parent,m)
    if len(bundles)<2:raise ValueError("Comparison needs at least two distinct completed run bundles")
    protocols={m["protocol_id"] for _,m in bundles.values()}
    if len(protocols)!=1:
        listing="\n".join(f"{m['model_info']['label']}: {m['protocol_id']}" for _,m in bundles.values())
        raise ValueError("Refusing to compare incompatible protocols. Use identical data, frozen seeds, mappings, analysis switches, and sampling configuration.\n"+listing)
    first=next(iter(bundles.values()))[1]
    if first["version"]!=VERSION or first["protocol"]["implementation_sha256"]!=IMPLEMENTATION_SHA256:
        raise ValueError("Run bundles were generated by a different evaluator version; use the matching notebook")
    if CONFIG["ranks"]!=first["protocol"]["config"]["ranks"]:
        raise ValueError("Comparison CONFIG['ranks'] must match the evaluated runs")
    fingerprints=None
    for folder,m in bundles.values():
        f=read_table(folder/"sampling_fingerprints.csv").sort_values(["dataset","split_seed","analysis"]).reset_index(drop=True)
        if fingerprints is None:fingerprints=f
        elif not fingerprints.equals(f):raise ValueError("Query/pair/geometry/gallery sampling differs between models")
    return bundles

def diagnostic_long(folder):
    rows=[]
    path=folder/"verification_results.csv"
    if path.is_file():
        f=read_table(path)
        keys=["dataset","split_seed","negative_kind","category_scope"]
        metrics=[c for c in f if c in {"AUROC","EER_interpolated","matched_positive_negative_gap","d_prime_descriptive"}
                 or c.startswith("TAR_at_FAR_")]
        for _,r in f.iterrows():
            for metric in metrics:
                value=pd.to_numeric(pd.Series([r[metric]]),errors="coerce").iloc[0]
                rows.append(dict(dataset=r.dataset,split_seed=r.split_seed,analysis="verification",
                                 condition=f"{r.negative_kind}/{r.category_scope}",metric=metric,value=value))
    path=folder/"geometry_results.csv"
    if path.is_file():
        f=read_table(path)
        exclude={"dataset","split_seed","membership_sha256","status","dimension","category_coverage","uniformity_t"}
        metrics=[c for c in f if c not in exclude and not c.startswith("n_") and not c.endswith("_samples")]
        for _,r in f.iterrows():
            for metric in metrics:
                value=pd.to_numeric(pd.Series([r[metric]]),errors="coerce").iloc[0]
                rows.append(dict(dataset=r.dataset,split_seed=r.split_seed,analysis="geometry",condition="original_space",
                                 metric=metric,value=value))
    return pd.DataFrame(rows)

def compare_runs(roots):
    bundles=comparison_manifests(roots)
    signature=digest({"runs":sorted(bundles),"bootstrap_replicates":CONFIG["bootstrap_replicates"],
                      "confidence_level":CONFIG["confidence_level"]})
    folder=OUTPUT_ROOT.resolve()/"comparisons"/signature[:12];folder.mkdir(parents=True,exist_ok=True)
    protocol=next(iter(bundles.values()))[1]["protocol"];datasets=protocol["datasets"]
    infos = {
        rid: info | MODEL_METADATA.get(info["model"], {})
        for rid, (_, manifest) in bundles.items()
        for info in (manifest["model_info"],)
    }
    baseline_losses = {"ArcFace", "Dynamic ArcFace", "SphereFace2"}
    contrasts = [
        (left, right)
        for left, source in infos.items()
        for right, baseline in infos.items()
        if source.get("loss") == "ApexID Hierarchical"
        and source.get("family") == baseline.get("family")
        and baseline.get("loss") in baseline_losses
    ]
    label_counts=Counter(i["label"] for i in infos.values())
    labels={rid:(i["label"] if label_counts[i["label"]]==1 else i["label"]+" / "+rid[:6]) for rid,i in infos.items()}
    all_summaries=[];all_macros=[];diag=[];metadata=[];queries={};curves=[];stress=[]
    for rid,(path,m) in bundles.items():
        extra=dict(run_id=rid,model=labels[rid],family=infos[rid].get("family","unspecified"),
                   loss=infos[rid].get("loss","unspecified"),training_seed=infos[rid].get("training_seed"),
                   ablation=infos[rid].get("ablation","unspecified"))
        metadata.append({**extra,"embedding_dimension":infos[rid]["embedding_dimension"],"source_folder":str(path)})
        all_summaries.append(read_table(path/"retrieval_summary.csv").assign(**extra))
        all_macros.append(read_table(path/"dataset_macro_summary.csv").assign(**extra))
        queries[rid]=pd.read_csv(path/"per_query_retrieval.csv.gz",keep_default_na=False)
        d=diagnostic_long(path)
        if len(d):diag.append(d.assign(**extra))
        curves.append(read_table(path/"cmc_curves.csv").assign(**extra))
        for name in ["few_shot_results","distractor_results"]:
            p=path/(name+".csv")
            if p.is_file():stress.append(read_table(p).assign(analysis=name,**extra))
    summaries=pd.concat(all_summaries,ignore_index=True);macros=pd.concat(all_macros,ignore_index=True)
    metadata=pd.DataFrame(metadata);save_table(folder/"models.csv",metadata)
    save_table(folder/"all_retrieval_summary.csv",summaries);save_table(folder/"all_dataset_macro.csv",macros)
    headline=macros.loc[macros.metric.isin(["mAP","rank1","identity_mAP","identity_rank1","mINP"])].pivot(
        index=["run_id","model"],columns="metric",values="mean").reset_index()
    save_table(folder/"headline_comparison.csv",headline)
    paired=[];identity_diffs=[]
    for left,right in contrasts:
        print(f"  Paired comparison: {labels[left]} minus {labels[right]}",flush=True)
        for condition in ["all"]:
            if condition not in set(queries[left].condition):continue
            ci=bootstrap_retrieval(queries[left],datasets,condition,other=queries[right])
            ci["model_a"]=labels[left];ci["model_b"]=labels[right];ci["run_a"]=left;ci["run_b"]=right
            ci["difference_definition"]="model_a minus model_b"
            ci["interval_excludes_zero"]=(ci.ci_low>0)|(ci.ci_high<0)
            paired.append(ci)
        a=queries[left].loc[queries[left].condition.eq("all")&queries[left].valid.astype(bool)].copy()
        b=queries[right].loc[queries[right].condition.eq("all")&queries[right].valid.astype(bool)].copy()
        for frame in [a,b]:frame["AP"]=pd.to_numeric(frame.AP,errors="raise")
        a=a.groupby(["dataset","split_seed","identity"]).AP.mean()
        b=b.groupby(["dataset","split_seed","identity"]).AP.mean()
        if not a.index.equals(b.index):raise ValueError("Identity groups differ in paired AP analysis")
        diff=(a-b).groupby(["dataset","identity"]).mean().rename("delta_AP").reset_index()
        diff=diff.assign(model_a=labels[left],model_b=labels[right],run_a=left,run_b=right)
        identity_diffs.append(diff)
        fig,ax=plt.subplots(figsize=(7,4))
        for ds,part in diff.loc[diff.dataset.isin(datasets)].groupby("dataset"):
            ax.hist(part.delta_AP,bins=25,alpha=.45,label=ds)
        ax.axvline(0,color="black",lw=1);ax.legend(fontsize=8)
        ax.set(xlabel="Per-identity AP difference (A − B)",ylabel="Identities",title=f"A: {labels[left]}\nB: {labels[right]}")
        save_figure(fig,folder/f"identity_AP_differences_{left[:7]}_{right[:7]}.png")
    paired=pd.concat(paired,ignore_index=True) if paired else pd.DataFrame()
    save_table(folder/"paired_retrieval_differences.csv",paired)
    save_table(folder/"per_identity_AP_differences.csv",pd.concat(identity_diffs,ignore_index=True) if identity_diffs else pd.DataFrame())
    dlong=pd.concat(diag,ignore_index=True) if diag else pd.DataFrame()
    if len(dlong):
        save_table(folder/"representation_all_splits.csv",dlong)
        dsumm=dlong.groupby(["run_id","model","dataset","analysis","condition","metric"],dropna=False).value.agg(
            mean="mean",split_sd="std",n_valid="count").reset_index()
        save_table(folder/"representation_summary.csv",dsumm)
        differences=[]
        keys=["dataset","analysis","condition","metric"]
        for left,right in contrasts:
            a=dsumm.loc[dsumm.run_id.eq(left)].set_index(keys)
            b=dsumm.loc[dsumm.run_id.eq(right)].set_index(keys)
            joined=a[["mean"]].join(b[["mean"]],lsuffix="_a",rsuffix="_b",how="outer")
            joined["difference"]=joined.mean_a-joined.mean_b
            differences.append(joined.reset_index().assign(model_a=labels[left],model_b=labels[right],
                               uncertainty="descriptive; no pair-independent significance claim"))
        save_table(folder/"representation_differences.csv",pd.concat(differences,ignore_index=True) if differences else pd.DataFrame())
    if stress:
        st=save_table(folder/"all_stress_tests.csv",pd.concat(stress,ignore_index=True))
        sr=[]
        for analysis,axis in [("few_shot_results","gallery_observations"),("distractor_results","n_distractor_identities")]:
            subset=st.loc[st.analysis.eq(analysis)].copy()
            if subset.empty or "mAP" not in subset:continue
            subset["mAP"]=pd.to_numeric(subset.mAP,errors="coerce")
            keys=["run_id","model","dataset","cohort",axis]
            if analysis=="distractor_results":keys.append("scope")
            su=subset.groupby(keys,dropna=False).mAP.agg(mean="mean",split_sd="std",n_valid_splits="count").reset_index()
            su["analysis"]=analysis;sr.append(su)
            cohort="common_max_k_cohort" if analysis=="few_shot_results" else "common_queries_across_counts"
            for ds in datasets:
                part=su.loc[su.dataset.eq(ds)&su.cohort.eq(cohort)]
                if part.empty:continue
                fig,ax=plt.subplots(figsize=(7,4))
                group_keys=["model"] if analysis=="few_shot_results" else ["model","scope"]
                for group,curve in part.groupby(group_keys,sort=True):
                    label=" / ".join(group) if isinstance(group,tuple) else str(group)
                    ax.plot(curve[axis],curve["mean"],marker="o",label=label)
                ax.set(xlabel=axis.replace("_"," "),ylabel="mAP",title=f"{ds} / {cohort}",ylim=(0,1.02));ax.legend(fontsize=7)
                save_figure(fig,folder/f"{analysis}_{safe_name(ds)}.png")
        if sr:save_table(folder/"stress_test_summary.csv",pd.concat(sr,ignore_index=True))
    c=pd.concat(curves,ignore_index=True)
    save_table(folder/"all_cmc_curves.csv",c)
    for ds in datasets:
        fig,ax=plt.subplots(figsize=(7,4))
        for model,part in c.loc[c.dataset.eq(ds)&c.condition.eq("all")].groupby("model",sort=True):
            means=part.groupby("rank").success.mean();ax.plot(means.index,means.values,label=model)
        ax.set(xlabel="Gallery image rank",ylabel="CMC success",title=ds,ylim=(0,1.02));ax.legend(fontsize=7)
        save_figure(fig,folder/f"cmc_{safe_name(ds)}.png")
        fig,axes=plt.subplots(1,2,figsize=(11,4));plotted=False
        first_seed=protocol["seeds"][0]
        for rid,(path,manifest) in bundles.items():
            rp=path/"splits"/ds/f"seed_{first_seed}"/"verification_ROC.csv.gz"
            if not rp.is_file():continue
            curve=read_table(rp)
            for ax,kind in zip(axes,["negative_all","negative_sibling"]):
                part=curve.loc[curve.negative_kind.eq(kind)]
                if len(part):ax.plot(np.maximum(part.FAR,1e-6),part.TAR,label=labels[rid]);plotted=True
                ax.set(xscale="log",xlabel="False acceptance rate",ylabel="True acceptance rate",title=kind,ylim=(0,1.02))
        if plotted:
            for ax in axes:
                if ax.get_legend_handles_labels()[0]:ax.legend(fontsize=7)
            fig.suptitle(f"{ds} / fixed split {first_seed}")
            save_figure(fig,folder/f"verification_{safe_name(ds)}.png")
        else:plt.close(fig)
    main=summaries.loc[summaries.condition.eq("all")&summaries.dataset.isin(datasets)&summaries.metric.eq("mAP")]
    heat=main.pivot(index="model",columns="dataset",values="mean").reindex(columns=datasets)
    fig,ax=plt.subplots(figsize=(max(7,len(datasets)*1.7),max(3,len(heat)*.45+1)))
    array=heat.to_numpy(float);im=ax.imshow(array,vmin=0,vmax=1,cmap="YlGnBu",aspect="auto")
    ax.set_xticks(np.arange(len(heat.columns)),heat.columns,rotation=20,ha="right")
    ax.set_yticks(np.arange(len(heat)),heat.index)
    for r in range(array.shape[0]):
        for col in range(array.shape[1]):
            ax.text(col,r,f"{100*array[r,col]:.1f}",ha="center",va="center",color="white" if array[r,col]>.65 else "black")
    fig.colorbar(im,ax=ax,label="mAP (fraction)");ax.set_title("Closed-set mAP; annotations are percentages")
    save_figure(fig,folder/"mAP_comparison.png")
    # Training-seed summaries are emitted only for explicit, distinct seed labels.
    seeded=macros.loc[macros.family.ne("unspecified")&macros.loss.ne("unspecified")&macros.training_seed.notna()].copy()
    training=[]
    if len(seeded):
        for key,part in seeded.groupby(["family","loss","ablation","metric"],dropna=False):
            if part.training_seed.duplicated().any():
                raise ValueError("Duplicate training-seed labels within a family/loss/ablation. Distinguish experiments in MODEL_METADATA")
            training.append(dict(zip(["family","loss","ablation","metric"],key),
                                 mean=float(part["mean"].mean()),training_seed_sd=float(part["mean"].std(ddof=1)),
                                 n_training_seeds=part.training_seed.nunique()))
    save_table(folder/"training_seed_summary.csv",training)
    (folder/"README.md").write_text("""# Comparing completed ApexID embedding evaluations

The protocol and every sampling membership fingerprint were checked before comparison.
Positive paired differences mean model A has the larger value. EER is lower-is-better;
geometry diagnostics do not all have a universally preferred direction.
The confidence intervals are percentile query-identity-block bootstrap intervals with
fixed galleries/checkpoints/datasets. Identity draws are shared across split seeds and
paired between models. No t-test treats the repeated splits as independent training runs.
No formal p-values or multiple-comparison-adjusted significance claims are produced.
Use predeclared primary comparisons; an interval excluding zero is descriptive evidence.

headline_comparison.csv: main dataset-macro retrieval scores.
paired_retrieval_differences.csv: per-dataset and macro paired differences with CIs.
representation_summary.csv / representation_differences.csv: diagnostic comparisons.
all_stress_tests.csv: limited-gallery and distractor results, with cohort/coverage metadata.
training_seed_summary.csv exists only when explicit training-seed provenance is supplied.
models.csv retains dimensions and declared family/loss/ablation labels. Cross-family
comparisons measure models; isolate loss effects within matched training families.
""",encoding="utf-8")
    write_json(folder/"comparison_manifest.json",dict(status="complete",runs=list(bundles),
               protocol_id=protocol["protocol_id"],implementation_sha256=IMPLEMENTATION_SHA256,
               confidence_level=CONFIG["confidence_level"],bootstrap_replicates=CONFIG["bootstrap_replicates"]))
    if MAKE_ZIP:shutil.make_archive(str(folder),"zip",root_dir=folder.parent,base_dir=folder.name)
    display(headline);print("Comparison saved:",folder,flush=True)
    return folder

## 4. Lightweight metric checks

These checks run on tiny synthetic arrays. They verify AP/mINP, stable tie handling,
positive exclusion, perfect/chance verification, and rejection of invalid embeddings.
They do not consume your real data or claim to validate a trained model's performance.

In [ ]:
def run_metric_self_checks():
    values, _ = ranked_values(np.array([.9,.8,.7,.6]), np.array([True,False,True,False]))
    assert np.isclose(values["AP"], (1 + 2/3)/2)
    assert np.isclose(values["INP"], 2/3) and values["rank1"] == 1
    values, _ = ranked_values(np.array([.5,.5,.5]), np.array([False,True,True]))
    assert values["first_rank"] == 2 and np.isclose(values["AP"], (1/2 + 2/3)/2)
    values, _ = ranked_values(np.array([.9,.8]), np.array([True,False]), np.array([False,True]))
    assert values is None
    pairs = pd.DataFrame({"query_id":["a","a","b","b"],"identity":["one","one","two","two"],
                          "pair_kind":["positive","negative_all"]*2,"cosine":[1.,0.,1.,0.]})
    result, _ = verification(pairs, "negative_all")
    assert result["AUROC"] == 1 and result["EER_interpolated"] == 0
    pairs["cosine"] = .5
    result, _ = verification(pairs, "negative_all")
    assert result["AUROC"] == .5 and result["EER_interpolated"] == .5
    assert result["TAR_at_FAR_0.01"] == 0
    try:
        normalized(np.zeros((2,3)))
    except ValueError:
        pass
    else:
        raise AssertionError("Zero vectors were not rejected")
    print("Metric self-checks passed.")

validate_configuration()
run_metric_self_checks()

## 5. Evaluate selected models

Leave `MODE="evaluate"` for separate model runs. If you are comparing completed runs,
this cell skips extraction-bundle discovery entirely.

The pipeline uses your external labels for scoring and descriptive diagnostics only.
It does not change losses, calibrators, split assignments, or embedding values on disk.

In [ ]:
COMPLETED_RUNS = []
if MODE in {"evaluate", "both"}:
    COMPLETED_RUNS = run_evaluation()
else:
    print("Evaluation skipped: comparison mode needs completed result bundles only.")

## 6. Compare all completed runs

Set `MODE="compare"` and list the saved run folders, parent folders, or individual ZIP files
in `COMPARE_ROOTS`. Directories should contain extracted `run_manifest.json` files; if all
you have is a ZIP, pass the ZIP path explicitly. No embeddings are loaded in this step.

All result bundles must come from this evaluator with the same evaluation configuration and
frozen data. A different backbone dimension is allowed and retained in `models.csv`; use
matched-family comparisons to make claims about the loss. A different metadata mapping,
split membership, sampling cap, or analysis switch is a different protocol and is refused.

`paired_retrieval_differences.csv` reports A minus B. Positive retrieval differences favor A.
The order of models does not imply a preferred baseline. Intervals excluding zero are reported
without a blanket statistical-significance claim across all tested comparisons.

In [ ]:
COMPARISON_FOLDER = None
if MODE in {"compare", "both"}:
    roots = COMPARE_ROOTS if COMPARE_ROOTS else COMPLETED_RUNS
    if not roots:
        raise ValueError("Set COMPARE_ROOTS to completed result folders or ZIPs.")
    with threadpool_limits(limits=CONFIG["numeric_threads"]):
        COMPARISON_FOLDER = compare_runs(roots)
else:
    print("Keep each result ZIP. Set MODE='compare' with COMPARE_ROOTS after evaluating the other models.")

## 7. Reading and reporting the outputs

| File | Purpose |
|---|---|
| `retrieval_results.csv` | Per-dataset/per-split metrics for full, within-category, and available metadata conditions |
| `retrieval_summary.csv` | Split means/SD with completeness counts |
| `dataset_macro_summary.csv` | Equal weight to the declared external datasets; pooled view excluded |
| `retrieval_confidence_intervals.csv` | Conditional query-identity-block intervals |
| `per_query_retrieval.csv.gz` | AP, INP, ranks, coverage, and exact positive/impostor margins |
| `verification_results.csv` | AUROC/EER/TAR@FAR, pair/anchor counts, category-specific sibling analysis |
| `pair_statistics.csv` | Identity-balanced similarity summaries and positive-pair alignment |
| `geometry_results.csv` | Original-space diagnostics with sample coverage |
| `few_shot_results.csv` | 1/2/5-observation gallery results for both eligibility schemes |
| `distractor_results.csv` | Sibling and cross-category distractor scaling |
| `analysis_status.csv` | What ran, what was unavailable, and why |
| `sampling_fingerprints.csv` | Proof that comparison runs used the same sampled memberships |
| `run_manifest.json` | Result status, protocol, provenance, environment, and artifact checksums |

Each split subfolder additionally contains pair scores, ROC points, top gallery matches,
geometry samples, covariance spectrum, stress-test query details, and PNG plots.

Comparison mode adds `headline_comparison.csv`, `paired_retrieval_differences.csv`,
`per_identity_AP_differences.csv`, `representation_summary.csv`,
`representation_differences.csv`, `stress_test_summary.csv`, and comparison figures.
`training_seed_summary.csv` is emitted only for explicitly labeled training seeds.

**Reporting guidance:** use mAP/Rank-1 and identity-balanced versions as primary evidence;
use mINP and same-category verification to support fine-grained identity discrimination.
Treat geometry as explanation, not a composite ranking score. These metrics are fractions;
multiply by 100 for percentage tables, and describe score differences as percentage points.
Do not combine split SD, conditional bootstrap uncertainty, and training-seed SD into one number.

**Ablations still needed for causal attribution:** evaluate matched checkpoints for global-only
sub-center identity loss; identity + category; identity + conditional; the full objective;
and the control that repeats only the extra positive penalty. Keep sub-center count fixed.
An optional randomized identity-to-category hierarchy control needs separate training.
Record variant names in `MODEL_METADATA[model]["ablation"]`; this notebook does not train those variants.

**Unavailable from one set of embeddings:** augmentation/occlusion robustness, original-versus-segmented
image changes, attribution maps/Score-CAM, loss-weight sweeps, and training-seed variability require
additional images, inference, or training runs. Missing metadata cannot establish cross-camera or
laterality invariance. Pretraining overlap must be audited from model/data provenance separately.

**Validation of this deliverable:** analytical metric checks, agreement with scikit-learn AP on
untied random rankings, and synthetic end-to-end evaluations for multiple models/datasets/seeds.
The real Stage 01 embedding arrays were not attached to the creation request, so actual ApexID
performance and runtime must be established when you run the notebook.

References:

- [Ye et al., Deep Learning for Person Re-identification: A Survey and Outlook](https://arxiv.org/abs/2001.04193) — mINP and retrieval evaluation.
- [Wang & Isola, Alignment and Uniformity on the Hypersphere](https://proceedings.mlr.press/v119/wang20k.html) — representation diagnostics.
- [Musgrave et al., A Metric Learning Reality Check](https://www.ecva.net/papers/eccv_2020/papers_ECCV/html/5170_ECCV_2020_paper.php) — controlled comparison methodology.
- [WildlifeReID-10k](https://arxiv.org/abs/2406.09211) — observation/time/similarity-aware evaluation motivation.